# Which Confidence Interval for Which Metric? — Companion Notebook

Reproducible companion notebook for

> A. Jaradat, A. Nasayreh, A. Bashkami, A. I. Alshdaifat and H. Gharaibeh,
> *Which Confidence Interval for Which Metric? A Coverage Study for Medical AI Classification*, CAISAIS 2026.

The saved outputs are those of the run reported in the paper: Google Colab, CPU-only runtime, Python 3.12.13, NumPy 2.0.2, SciPy 1.16.3, scikit-learn 1.6.1, pandas 2.2.2, PyTorch 2.11.0 (CPU build).

### Contents
| Section | Experiment | Notebook output | Paper |
|---|---|---|---|
| 2–6 | Breast Cancer Wisconsin (n_test = 171), five classifiers | Tables 1–4 | boundary collapse and ceiling effect (Sec. V-D) |
| 7–11 | PneumoniaMNIST (n_test = 624), CNN and pixel logistic regression | Tables 5–6 | Table VIII, Sec. V-D |
| 12 | Monte-Carlo coverage for accuracy (eight constructions) | Table 7 | Table V |
| 13 | Wall-clock cost of B = 10,000 resamples | Table 8 | Sec. IV |
| 14–15 | Figures and export | `results/` | — |
| 16 | Coverage for F1, MCC, balanced accuracy and AUROC; paired differences; real-data paired comparisons | Tables 9–11, Fig. 8 | Tables IV, VI–VIII, Fig. 1 |

Settings S3 and S4 of Table IV are run by `scripts/sensitivity_check.py`, and Fig. 1 of the paper is drawn from Table 9 by `scripts/make_fig_coverage.py`.

### How to run
1. Open the notebook in Google Colab; a CPU runtime is sufficient (the reported run used no GPU).
2. `Runtime → Run all`. On a CPU runtime the full run takes about 20–25 min (CNN training ≈ 6.5 min, Section 12 ≈ 2 min, Section 16 ≈ 10 min).
3. For a short smoke test, set `CONFIG["FAST_MODE"] = True` in Section 0.
4. Sections 15 and 16 write all tables, figures and `results_all.json` to `results/` and zip the folder.

> **Methodological note.** The resampling unit is the *patient/image*: the trained model is held fixed and only finite-test-set sampling error is quantified. This is the quantity a clinician needs ("would this accuracy hold on the next cohort?"). It does **not** capture training-set variability, which requires repeated retraining and is discussed as a limitation.

In [1]:
# =============================================================================
# 0 · Environment, dependencies, global configuration
# =============================================================================
import importlib, os, subprocess, sys, time, json, random, warnings
from pathlib import Path

def _ensure(module: str, pip_name: str = None):
    """Install a package only if it is not already importable (Colab-friendly)."""
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module],
                       check=True)

for mod, pip in [("medmnist", "medmnist"), ("torch", "torch"), ("sklearn", "scikit-learn")]:
    _ensure(mod, pip)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import sklearn
import scipy

# -----------------------------------------------------------------------------
# Master configuration — every stochastic component reads from here.
# -----------------------------------------------------------------------------
CONFIG = {
    "SEED":          42,
    "FAST_MODE":     False,      # True -> tiny budgets for a smoke test
    # --- bootstrap
    "B":             10_000,     # resamples (paper value)
    "ALPHA":         0.05,       # -> 95% CI
    "CI_METHOD":     "bca",      # {"percentile", "basic", "bca"}
    "STRATIFIED":    False,      # stratified-by-class resampling
    "CHUNK":         2_000,      # resamples held in RAM at once
    # --- Experiment 1 (tabular)
    "TAB_TEST_SIZE": 0.30,
    # --- Experiment 2 (medical imaging)
    "MED_FLAG":      "pneumoniamnist",   # any binary MedMNIST 2D flag
    "IMG_SIZE":      28,
    "EPOCHS":        15,
    "BATCH":         128,
    "LR":            1e-3,
    "WD":            1e-4,
    "WIDTH":         32,         # CNN base channel width
    # --- sample-size sweep
    "SWEEP_SIZES":   [50, 100, 200, 400, None],   # None = full test set
    "SWEEP_REPS":    10,         # independent sub-cohorts per size
    # --- Experiment 3 (coverage)
    "COV_MC":        2_000,      # Monte-Carlo replications
    "COV_B":         2_000,      # bootstrap resamples inside each replication
    "COV_P":         [0.80, 0.90, 0.95],
    "COV_N":         [30, 50, 100, 200, 500],
    # --- I/O
    "OUTDIR":        "results",
}

if CONFIG["FAST_MODE"]:
    CONFIG.update(B=2_000, EPOCHS=3, SWEEP_REPS=3, COV_MC=300, COV_B=500,
                  COV_N=[50, 200], COV_P=[0.90])

OUT = Path(CONFIG["OUTDIR"]); (OUT / "figures").mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Publication-grade figure defaults (IEEE single-column = 3.5 in).
matplotlib.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "serif", "font.size": 9, "axes.titlesize": 10,
    "axes.labelsize": 9, "legend.fontsize": 8, "axes.grid": True,
    "grid.alpha": 0.3, "grid.linewidth": 0.5, "axes.axisbelow": True,
})

ENV = {
    "python": sys.version.split()[0], "numpy": np.__version__,
    "scipy": scipy.__version__, "sklearn": sklearn.__version__,
    "torch": torch.__version__, "pandas": pd.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
}
print("Environment (record this in the manuscript's reproducibility statement):")
for k, v in ENV.items():
    print(f"  {k:9s}: {v}")
print(f"\nFAST_MODE = {CONFIG['FAST_MODE']}   B = {CONFIG['B']}   seed = {CONFIG['SEED']}")

RESULTS = {"config": {k: v for k, v in CONFIG.items()}, "environment": ENV}

Environment (record this in the manuscript's reproducibility statement):
  python   : 3.12.13
  numpy    : 2.0.2
  scipy    : 1.16.3
  sklearn  : 1.6.1
  torch    : 2.11.0+cpu
  pandas   : 2.2.2
  device   : cpu
  gpu      : none

FAST_MODE = False   B = 10000   seed = 42


In [2]:
%%writefile bootci.py
"""
bootci.py
=========
Vectorised non-parametric bootstrap confidence intervals (CIs) for the
evaluation of medical machine-learning models.

Design goals
------------
1. *Correctness*  : every metric is computed with an explicit, tie-aware,
                    fully vectorised NumPy implementation that is unit-tested
                    against scikit-learn reference implementations.
2. *Scalability*  : B = 10,000 resamples are generated in memory-bounded
                    chunks, so the peak RAM footprint is O(chunk * n) and not
                    O(B * n).
3. *Reproducibility*: every stochastic routine takes an explicit `seed`, and
                    `numpy.random.Generator` (PCG64) is used rather than the
                    deprecated global legacy RandomState.
4. *Auditability* : classical closed-form intervals (Wald, Wilson,
                    Agresti-Coull, Clopper-Pearson, Jeffreys) and the
                    asymptotic DeLong AUC interval are included so that the
                    bootstrap can be benchmarked instead of merely asserted.

Complexity (n = test-set size, B = number of resamples)
-------------------------------------------------------
  index generation ............ time O(B n)          space O(chunk * n)
  threshold metrics (ACC, F1..) time O(B n)          space O(chunk * n)
  AUROC (tie-aware midranks) .. time O(B n log n)    space O(chunk * n)
  BCa jackknife ............... time O(n^2) [O(n) evaluations of an O(n) metric]
  DeLong AUC variance ......... time O(n log n)      space O(n)

Authors : A. Jaradat, A. Nasayreh, A. Bashkami, A. I. Alshdaifat, H. Gharaibeh
          (companion code for the CAISAIS 2026 paper).
Licence : MIT.
"""
from __future__ import annotations

import warnings
from dataclasses import dataclass, field
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
from scipy import stats

__all__ = [
    "set_global_seed",
    "BootResult",
    "bootstrap_ci",
    "bootstrap_many",
    "paired_bootstrap_diff",
    "mcnemar_exact",
    "binomial_ci",
    "rate_ci_closed_form",
    "delong_auc_ci",
    "coverage_simulation",
    "format_ci",
    "METRICS",
]

# --------------------------------------------------------------------------- #
# 0. Reproducibility
# --------------------------------------------------------------------------- #
def set_global_seed(seed: int = 42, deterministic_torch: bool = True) -> None:
    """Seed every RNG that can influence the reported numbers.

    Torch is imported lazily so that the module remains usable in a
    NumPy-only environment.
    """
    import os
    import random

    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)  # only affects legacy global calls; Generators are explicit
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        if deterministic_torch:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
            # warn_only=True: a few cuDNN kernels have no deterministic variant.
            torch.use_deterministic_algorithms(True, warn_only=True)
            os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    except ImportError:
        pass


# --------------------------------------------------------------------------- #
# 1. Tie-aware vectorised midranks  (needed for a correct bootstrap AUROC)
# --------------------------------------------------------------------------- #
def _midranks_2d(a: np.ndarray) -> np.ndarray:
    """Average ("mid") ranks of every row of `a`, ties resolved by averaging.

    Equivalent to ``scipy.stats.rankdata(a, method='average', axis=1)`` but
    written explicitly because ties are *guaranteed* under resampling with
    replacement (a duplicated observation produces an exactly duplicated
    score) and the tie correction is what makes the Mann-Whitney identity
    AUC = (R+ - n+(n+ +1)/2) / (n+ n-) exact.

    Complexity: O(B n log n) time, O(B n) space.
    """
    B, n = a.shape
    order = np.argsort(a, axis=1, kind="stable")
    s = np.take_along_axis(a, order, axis=1)

    pos = np.broadcast_to(np.arange(n), (B, n))

    starts = np.ones((B, n), dtype=bool)
    starts[:, 1:] = s[:, 1:] != s[:, :-1]
    ends = np.ones((B, n), dtype=bool)
    ends[:, :-1] = s[:, 1:] != s[:, :-1]

    first = np.maximum.accumulate(np.where(starts, pos, 0), axis=1)
    last = np.minimum.accumulate(np.where(ends, pos, n - 1)[:, ::-1], axis=1)[:, ::-1]

    ranks_sorted = (first + last) / 2.0 + 1.0  # 1-based average ranks
    ranks = np.empty((B, n), dtype=np.float64)
    np.put_along_axis(ranks, order, ranks_sorted, axis=1)
    return ranks


# --------------------------------------------------------------------------- #
# 2. Vectorised metric kernels
#    Every kernel receives the *resampled* matrices
#       yt : (B, n) int   true labels in {0, 1}
#       yp : (B, n) int   hard predictions in {0, 1}
#       ys : (B, n) float positive-class scores
#    and returns a (B,) vector.  Degenerate resamples yield np.nan.
# --------------------------------------------------------------------------- #
def _confusion(yt: np.ndarray, yp: np.ndarray):
    tp = np.sum((yt == 1) & (yp == 1), axis=1).astype(np.float64)
    tn = np.sum((yt == 0) & (yp == 0), axis=1).astype(np.float64)
    fp = np.sum((yt == 0) & (yp == 1), axis=1).astype(np.float64)
    fn = np.sum((yt == 1) & (yp == 0), axis=1).astype(np.float64)
    return tp, tn, fp, fn


def _safe_div(num: np.ndarray, den: np.ndarray) -> np.ndarray:
    out = np.full_like(num, np.nan, dtype=np.float64)
    ok = den > 0
    out[ok] = num[ok] / den[ok]
    return out


def _accuracy(yt, yp, ys):
    return np.mean(yt == yp, axis=1).astype(np.float64)


def _sensitivity(yt, yp, ys):          # recall / TPR
    tp, tn, fp, fn = _confusion(yt, yp)
    return _safe_div(tp, tp + fn)


def _specificity(yt, yp, ys):          # TNR
    tp, tn, fp, fn = _confusion(yt, yp)
    return _safe_div(tn, tn + fp)


def _ppv(yt, yp, ys):                  # precision
    tp, tn, fp, fn = _confusion(yt, yp)
    return _safe_div(tp, tp + fp)


def _npv(yt, yp, ys):
    tp, tn, fp, fn = _confusion(yt, yp)
    return _safe_div(tn, tn + fn)


def _f1(yt, yp, ys):
    tp, tn, fp, fn = _confusion(yt, yp)
    return _safe_div(2 * tp, 2 * tp + fp + fn)


def _balanced_accuracy(yt, yp, ys):
    return 0.5 * (_sensitivity(yt, yp, ys) + _specificity(yt, yp, ys))


def _mcc(yt, yp, ys):
    tp, tn, fp, fn = _confusion(yt, yp)
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return _safe_div(num, den)


def _auroc(yt, yp, ys):
    """Tie-corrected AUROC via the Mann-Whitney U identity."""
    ranks = _midranks_2d(ys)
    n_pos = np.sum(yt == 1, axis=1).astype(np.float64)
    n_neg = np.sum(yt == 0, axis=1).astype(np.float64)
    r_pos = np.sum(ranks * (yt == 1), axis=1)
    num = r_pos - n_pos * (n_pos + 1.0) / 2.0
    return _safe_div(num, n_pos * n_neg)


METRICS: Dict[str, Callable] = {
    "accuracy": _accuracy,
    "sensitivity": _sensitivity,
    "specificity": _specificity,
    "ppv": _ppv,
    "npv": _npv,
    "f1": _f1,
    "balanced_accuracy": _balanced_accuracy,
    "mcc": _mcc,
    "auroc": _auroc,
}

NEEDS_SCORES = {"auroc"}


# --------------------------------------------------------------------------- #
# 3. Resampling engine
# --------------------------------------------------------------------------- #
def _iter_index_chunks(
    n: int,
    B: int,
    rng: np.random.Generator,
    strata: Optional[np.ndarray] = None,
    chunk_size: int = 2000,
) -> Iterable[np.ndarray]:
    """Yield (chunk, n) matrices of resampling indices.

    strata is None -> i.i.d. bootstrap: indices ~ Uniform{0..n-1}.
    strata given   -> stratified bootstrap: resample *within* each class, so
                      the class prevalence of every replicate equals that of
                      the observed test set.  This removes the extra variance
                      caused by fluctuating prevalence and prevents degenerate
                      single-class replicates when the minority class is tiny.
                      NOTE: it conditions on the observed prevalence, hence it
                      answers "what if we re-drew patients within each arm"
                      rather than "what if we re-drew the cohort".  Report the
                      scheme explicitly in the manuscript.
    """
    if strata is not None:
        groups = [np.flatnonzero(strata == g) for g in np.unique(strata)]
    done = 0
    while done < B:
        b = int(min(chunk_size, B - done))
        if strata is None:
            idx = rng.integers(0, n, size=(b, n), dtype=np.int64)
        else:
            idx = np.empty((b, n), dtype=np.int64)
            col = 0
            for gi in groups:
                k = gi.size
                idx[:, col : col + k] = gi[rng.integers(0, k, size=(b, k))]
                col += k
        yield idx
        done += b


def _replicates(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_score: Optional[np.ndarray],
    metric: str,
    B: int,
    rng: np.random.Generator,
    stratified: bool,
    chunk_size: int,
) -> np.ndarray:
    fn = METRICS[metric]
    n = y_true.size
    strata = y_true if stratified else None
    out = np.empty(B, dtype=np.float64)
    pos = 0
    ys_src = y_score if y_score is not None else np.zeros(n)
    for idx in _iter_index_chunks(n, B, rng, strata, chunk_size):
        b = idx.shape[0]
        out[pos : pos + b] = fn(y_true[idx], y_pred[idx], ys_src[idx])
        pos += b
    return out


# --------------------------------------------------------------------------- #
# 4. Interval constructors
# --------------------------------------------------------------------------- #
def _percentile_ci(theta_star: np.ndarray, alpha: float) -> Tuple[float, float]:
    lo, hi = np.nanpercentile(theta_star, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)


def _basic_ci(theta_hat: float, theta_star: np.ndarray, alpha: float) -> Tuple[float, float]:
    lo, hi = _percentile_ci(theta_star, alpha)
    return float(2 * theta_hat - hi), float(2 * theta_hat - lo)


def _bca_ci(
    theta_hat: float,
    theta_star: np.ndarray,
    jack: np.ndarray,
    alpha: float,
    mid_p: bool = False,
) -> Tuple[float, float, float, float]:
    """Bias-corrected and accelerated interval (Efron, 1987).

    Returns (lo, hi, z0, a).

    `mid_p=False` reproduces the textbook definition
        z0 = Phi^-1( #{theta* < theta_hat} / B ).
    For discrete metrics on small test sets the bootstrap distribution is
    lattice-valued and a large point mass sits exactly at theta_hat; the
    mid-p variant  ( #{<} + 0.5 #{=} ) / B  is then markedly better behaved.
    Both are exposed; the manuscript must state which one was used.
    """
    ts = theta_star[np.isfinite(theta_star)]
    B = ts.size
    if B == 0:
        return np.nan, np.nan, np.nan, np.nan

    n_less = float(np.sum(ts < theta_hat))
    if mid_p:
        n_less += 0.5 * float(np.sum(ts == theta_hat))
    prop = n_less / B
    prop = min(max(prop, 0.5 / B), 1.0 - 0.5 / B)  # keep z0 finite
    z0 = float(stats.norm.ppf(prop))

    jk = jack[np.isfinite(jack)]
    jbar = jk.mean()
    d = jbar - jk
    denom = 6.0 * (np.sum(d**2) ** 1.5)
    a = float(np.sum(d**3) / denom) if denom > 0 else 0.0

    z_lo, z_hi = stats.norm.ppf(alpha / 2), stats.norm.ppf(1 - alpha / 2)
    def _adj(z):
        val = z0 + (z0 + z) / (1.0 - a * (z0 + z))
        return float(stats.norm.cdf(val))

    a1, a2 = _adj(z_lo), _adj(z_hi)
    if not np.isfinite(a1) or not np.isfinite(a2) or a1 >= a2:
        warnings.warn("BCa adjustment degenerate; falling back to percentile.")
        lo, hi = _percentile_ci(ts, alpha)
        return lo, hi, z0, a
    lo, hi = np.percentile(ts, [100 * a1, 100 * a2])
    return float(lo), float(hi), z0, a


def _jackknife(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_score: Optional[np.ndarray],
    metric: str,
    max_n: int = 5000,
) -> np.ndarray:
    """Leave-one-out replicates required by the BCa acceleration constant."""
    n = y_true.size
    if n > max_n:
        warnings.warn(f"n={n} > max_n={max_n}: jackknife skipped, BCa unavailable.")
        return np.full(1, np.nan)
    fn = METRICS[metric]
    base = np.arange(n)
    # (n, n-1) leave-one-out index matrix; 8 bytes * n * (n-1) — fine for n <= 5000.
    loo = np.stack([np.delete(base, i) for i in range(n)])
    ys_src = y_score if y_score is not None else np.zeros(n)
    return fn(y_true[loo], y_pred[loo], ys_src[loo])


# --------------------------------------------------------------------------- #
# 5. Public API
# --------------------------------------------------------------------------- #
@dataclass
class BootResult:
    metric: str
    estimate: float
    ci_low: float
    ci_high: float
    method: str
    alpha: float
    B: int
    n: int
    se_boot: float
    bias_boot: float
    n_valid: int
    stratified: bool
    boundary_degenerate: bool = False
    z0: Optional[float] = None
    accel: Optional[float] = None
    replicates: Optional[np.ndarray] = field(default=None, repr=False)

    @property
    def width(self) -> float:
        return self.ci_high - self.ci_low

    @property
    def moe(self) -> float:
        """Half-width, i.e. the margin of error in +/- notation."""
        return 0.5 * (self.ci_high - self.ci_low)

    def as_row(self) -> Dict:
        return {
            "metric": self.metric,
            "estimate": self.estimate,
            "ci_low": self.ci_low,
            "ci_high": self.ci_high,
            "width": self.width,
            "moe": self.moe,
            "se_boot": self.se_boot,
            "bias_boot": self.bias_boot,
            "method": self.method,
            "B": self.B,
            "n": self.n,
            "n_valid": self.n_valid,
            "stratified": self.stratified,
            "boundary_degenerate": self.boundary_degenerate,
            "z0": self.z0,
            "accel": self.accel,
        }

    def __str__(self) -> str:
        return format_ci(self.estimate, self.ci_low, self.ci_high)


def format_ci(est: float, lo: float, hi: float, digits: int = 4) -> str:
    return f"{est:.{digits}f} [{lo:.{digits}f}, {hi:.{digits}f}]"


def bootstrap_ci(
    y_true: Sequence,
    y_pred: Sequence,
    y_score: Optional[Sequence] = None,
    metric: str = "accuracy",
    B: int = 10_000,
    alpha: float = 0.05,
    method: str = "bca",           # {"percentile", "basic", "bca"}
    seed: int = 42,
    stratified: bool = False,
    chunk_size: int = 2000,
    mid_p: bool = False,
    return_replicates: bool = True,
) -> BootResult:
    """Non-parametric bootstrap CI for a single metric on a fixed test set.

    The resampling unit is the *patient/image*, i.e. the model is held fixed
    and only the finite-test-set sampling error is quantified.  This is the
    quantity a clinician needs when asking "would this accuracy hold on the
    next cohort drawn from the same population?"  It does *not* capture
    training-set variability (see `Discussion` in the manuscript).
    """
    y_true = np.asarray(y_true).ravel().astype(np.int64)
    y_pred = np.asarray(y_pred).ravel().astype(np.int64)
    y_score = None if y_score is None else np.asarray(y_score, dtype=np.float64).ravel()

    if metric not in METRICS:
        raise KeyError(f"unknown metric '{metric}'; available: {sorted(METRICS)}")
    if metric in NEEDS_SCORES and y_score is None:
        raise ValueError(f"metric '{metric}' requires y_score")
    if not (y_true.size == y_pred.size and (y_score is None or y_score.size == y_true.size)):
        raise ValueError("y_true, y_pred and y_score must have identical length")

    n = y_true.size
    rng = np.random.default_rng(seed)
    fn = METRICS[metric]
    theta_hat = float(fn(y_true[None, :], y_pred[None, :],
                         (y_score if y_score is not None else np.zeros(n))[None, :])[0])

    ts = _replicates(y_true, y_pred, y_score, metric, B, rng, stratified, chunk_size)
    valid = np.isfinite(ts)
    n_valid = int(valid.sum())
    if n_valid < 0.9 * B:
        warnings.warn(
            f"{B - n_valid}/{B} degenerate replicates for metric '{metric}'. "
            "Consider stratified=True (small minority class)."
        )

    # --- Failure mode: if the observed statistic sits exactly on the boundary
    # of the parameter space (e.g. specificity = 1.000 because no negative was
    # misclassified), every bootstrap replicate is identical and the interval
    # collapses to zero width.  The bootstrap is then NOT valid: the empirical
    # distribution carries no information about how far below 1 the true value
    # may lie.  Report `rate_ci_closed_form(..., method='clopper_pearson')`
    # instead (see Section "Boundary failure" of the manuscript).
    boundary = bool(np.nanstd(ts) < 1e-12) or theta_hat in (0.0, 1.0)
    if boundary:
        warnings.warn(
            f"metric '{metric}' = {theta_hat:.4f} lies on the boundary of the "
            "parameter space: the non-parametric bootstrap degenerates "
            "(zero-width interval). Use rate_ci_closed_form() / Clopper-Pearson."
        )

    z0 = accel = None
    if method == "percentile":
        lo, hi = _percentile_ci(ts, alpha)
    elif method == "basic":
        lo, hi = _basic_ci(theta_hat, ts, alpha)
    elif method == "bca":
        jack = _jackknife(y_true, y_pred, y_score, metric)
        if np.all(np.isnan(jack)):
            lo, hi = _percentile_ci(ts, alpha)
            method = "percentile(bca-fallback)"
        else:
            lo, hi, z0, accel = _bca_ci(theta_hat, ts, jack, alpha, mid_p=mid_p)
    else:
        raise KeyError("method must be one of {'percentile', 'basic', 'bca'}")

    return BootResult(
        metric=metric,
        estimate=theta_hat,
        ci_low=lo,
        ci_high=hi,
        method=method,
        alpha=alpha,
        B=B,
        n=n,
        se_boot=float(np.nanstd(ts, ddof=1)),
        bias_boot=float(np.nanmean(ts) - theta_hat),
        n_valid=n_valid,
        stratified=stratified,
        boundary_degenerate=boundary,
        z0=z0,
        accel=accel,
        replicates=ts if return_replicates else None,
    )


def bootstrap_many(
    y_true, y_pred, y_score=None, metrics: Sequence[str] = ("accuracy",), **kw
) -> Dict[str, BootResult]:
    """Convenience wrapper: identical seed => identical resamples across metrics,
    which keeps the reported intervals mutually consistent."""
    return {m: bootstrap_ci(y_true, y_pred, y_score, metric=m, **kw) for m in metrics}


# --------------------------------------------------------------------------- #
# 6. Paired comparison of two models on the SAME test set
# --------------------------------------------------------------------------- #
def paired_bootstrap_diff(
    y_true: Sequence,
    pred_a: Sequence,
    pred_b: Sequence,
    score_a: Optional[Sequence] = None,
    score_b: Optional[Sequence] = None,
    metric: str = "accuracy",
    B: int = 10_000,
    alpha: float = 0.05,
    seed: int = 42,
    stratified: bool = False,
    chunk_size: int = 2000,
) -> Dict:
    """CI and achieved significance level (ASL) for Delta = metric(A) - metric(B).

    The *same* resampled indices are applied to both models, so the strong
    positive correlation between two models evaluated on one cohort is
    preserved.  Ignoring the pairing (comparing two independent CIs and
    checking for overlap) is a well-known conservative error: non-overlapping
    intervals imply significance, but overlapping intervals do NOT imply
    non-significance.
    """
    y_true = np.asarray(y_true).ravel().astype(np.int64)
    pred_a = np.asarray(pred_a).ravel().astype(np.int64)
    pred_b = np.asarray(pred_b).ravel().astype(np.int64)
    n = y_true.size
    zeros = np.zeros(n)
    sa = zeros if score_a is None else np.asarray(score_a, float).ravel()
    sb = zeros if score_b is None else np.asarray(score_b, float).ravel()
    if metric in NEEDS_SCORES and (score_a is None or score_b is None):
        raise ValueError(f"metric '{metric}' requires score_a and score_b")

    fn = METRICS[metric]
    theta_a = float(fn(y_true[None, :], pred_a[None, :], sa[None, :])[0])
    theta_b = float(fn(y_true[None, :], pred_b[None, :], sb[None, :])[0])
    delta_hat = theta_a - theta_b

    rng = np.random.default_rng(seed)
    strata = y_true if stratified else None
    d = np.empty(B, dtype=np.float64)
    pos = 0
    for idx in _iter_index_chunks(n, B, rng, strata, chunk_size):
        b = idx.shape[0]
        yt = y_true[idx]
        d[pos : pos + b] = fn(yt, pred_a[idx], sa[idx]) - fn(yt, pred_b[idx], sb[idx])
        pos += b

    dv = d[np.isfinite(d)]
    lo, hi = np.percentile(dv, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    # Two-sided ASL under H0: Delta = 0, obtained by centring the bootstrap
    # distribution (Efron & Tibshirani, 1993, ch. 16).
    asl = float(np.mean(np.abs(dv - delta_hat) >= abs(delta_hat)))
    return {
        "metric": metric,
        "theta_a": theta_a,
        "theta_b": theta_b,
        "delta": delta_hat,
        "ci_low": float(lo),
        "ci_high": float(hi),
        "asl_p": asl,
        "significant_at_alpha": bool(lo > 0 or hi < 0),
        "B": B,
        "n": n,
        "replicates": dv,
    }


def mcnemar_exact(y_true, pred_a, pred_b) -> Dict:
    """Exact (binomial) McNemar test — closed-form cross-check for the paired
    bootstrap on *accuracy*.  Discordant pairs only."""
    y_true = np.asarray(y_true).ravel()
    a_ok = np.asarray(pred_a).ravel() == y_true
    b_ok = np.asarray(pred_b).ravel() == y_true
    n01 = int(np.sum(a_ok & ~b_ok))   # A right, B wrong
    n10 = int(np.sum(~a_ok & b_ok))   # A wrong, B right
    if n01 + n10 == 0:
        return {"n01": n01, "n10": n10, "p_value": 1.0}
    p = stats.binomtest(n01, n01 + n10, 0.5, alternative="two-sided").pvalue
    return {"n01": n01, "n10": n10, "p_value": float(p)}


# --------------------------------------------------------------------------- #
# 7. Closed-form binomial intervals (benchmarks for the bootstrap)
# --------------------------------------------------------------------------- #
def binomial_ci(k: int, n: int, alpha: float = 0.05, method: str = "wilson") -> Tuple[float, float]:
    """Interval for a binomial proportion.

    method in {"wald", "wilson", "wilson_cc", "agresti_coull",
               "clopper_pearson", "jeffreys"}.
    Wald is included *only* as the negative control criticised in the paper:
    its coverage collapses when n is small or p approaches 0/1, and it can
    return limits outside [0, 1] (Brown, Cai & DasGupta, 2001).
    """
    if n <= 0:
        raise ValueError("n must be positive")
    p = k / n
    z = stats.norm.ppf(1 - alpha / 2)

    if method == "wald":
        se = np.sqrt(p * (1 - p) / n)
        return p - z * se, p + z * se           # deliberately NOT clipped
    if method in ("wilson", "wilson_cc"):
        cc = 0.5 / n if method == "wilson_cc" else 0.0
        d = 1 + z**2 / n
        centre = (p + z**2 / (2 * n)) / d
        half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2) + cc / n) / d
        return max(0.0, centre - half), min(1.0, centre + half)
    if method == "agresti_coull":
        n_t = n + z**2
        p_t = (k + z**2 / 2) / n_t
        half = z * np.sqrt(p_t * (1 - p_t) / n_t)
        return max(0.0, p_t - half), min(1.0, p_t + half)
    if method == "clopper_pearson":
        lo = 0.0 if k == 0 else stats.beta.ppf(alpha / 2, k, n - k + 1)
        hi = 1.0 if k == n else stats.beta.ppf(1 - alpha / 2, k + 1, n - k)
        return float(lo), float(hi)
    if method == "jeffreys":
        lo = 0.0 if k == 0 else stats.beta.ppf(alpha / 2, k + 0.5, n - k + 0.5)
        hi = 1.0 if k == n else stats.beta.ppf(1 - alpha / 2, k + 0.5, n - k + 0.5)
        return float(lo), float(hi)
    raise KeyError(f"unknown method '{method}'")


RATE_DENOMINATOR = {
    "accuracy": "all",
    "sensitivity": "true_pos",
    "specificity": "true_neg",
    "ppv": "pred_pos",
    "npv": "pred_neg",
}


def rate_ci_closed_form(
    y_true, y_pred, metric: str = "accuracy", alpha: float = 0.05,
    method: str = "clopper_pearson"
) -> Dict:
    """Closed-form binomial interval for any metric that is a simple rate.

    Each of accuracy / sensitivity / specificity / PPV / NPV is a proportion
    k/m over an explicitly defined denominator, so an exact or score interval
    applies directly and — unlike the non-parametric bootstrap — remains valid
    when the observed rate is exactly 0 or 1.

    Caveat: for PPV and NPV the denominator (number of predicted positives /
    negatives) is itself random; conditioning on it is the standard, mildly
    anti-conservative convention and must be stated in the manuscript.
    """
    y_true = np.asarray(y_true).ravel().astype(int)
    y_pred = np.asarray(y_pred).ravel().astype(int)
    if metric not in RATE_DENOMINATOR:
        raise KeyError(f"'{metric}' is not a simple rate; use bootstrap_ci().")
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    table = {
        "accuracy": (tp + tn, tp + tn + fp + fn),
        "sensitivity": (tp, tp + fn),
        "specificity": (tn, tn + fp),
        "ppv": (tp, tp + fp),
        "npv": (tn, tn + fn),
    }
    k, m = table[metric]
    if m == 0:
        return {"metric": metric, "k": k, "n": m, "estimate": np.nan,
                "ci_low": np.nan, "ci_high": np.nan, "method": method}
    lo, hi = binomial_ci(k, m, alpha=alpha, method=method)
    return {"metric": metric, "k": k, "n": m, "estimate": k / m,
            "ci_low": float(lo), "ci_high": float(hi), "method": method}


# --------------------------------------------------------------------------- #
# 8. DeLong asymptotic AUC interval (parametric benchmark)
# --------------------------------------------------------------------------- #
def _midrank_1d(x: np.ndarray) -> np.ndarray:
    return stats.rankdata(x, method="average")


def delong_auc_ci(y_true, y_score, alpha: float = 0.05) -> Dict:
    """AUC, its DeLong variance and the corresponding normal interval.

    Implementation follows the O(n log n) formulation of Sun & Xu (2014) for
    the single-classifier case, using the structural components (placement
    values) of DeLong, DeLong & Clarke-Pearson (1988).
    """
    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score, dtype=float).ravel()
    pos = y_score[y_true == 1]
    neg = y_score[y_true == 0]
    m, n = pos.size, neg.size
    if m == 0 or n == 0:
        return {"auc": np.nan, "var": np.nan, "ci_low": np.nan, "ci_high": np.nan}

    tz = _midrank_1d(np.concatenate([pos, neg]))
    tx, ty = _midrank_1d(pos), _midrank_1d(neg)
    auc = (tz[:m].sum() - m * (m + 1) / 2.0) / (m * n)

    v01 = (tz[:m] - tx) / n            # placement values for positives
    v10 = 1.0 - (tz[m:] - ty) / m      # placement values for negatives
    s01 = np.var(v01, ddof=1) if m > 1 else 0.0
    s10 = np.var(v10, ddof=1) if n > 1 else 0.0
    var = s01 / m + s10 / n
    se = float(np.sqrt(max(var, 0.0)))
    z = stats.norm.ppf(1 - alpha / 2)
    return {
        "auc": float(auc),
        "var": float(var),
        "se": se,
        "ci_low": float(auc - z * se),
        "ci_high": float(auc + z * se),
    }


# --------------------------------------------------------------------------- #
# 9. Monte-Carlo coverage study
# --------------------------------------------------------------------------- #
def coverage_simulation(
    p_true: float,
    n: int,
    n_mc: int = 2000,
    B: int = 2000,
    alpha: float = 0.05,
    seed: int = 0,
    methods: Sequence[str] = ("wald", "wilson", "clopper_pearson",
                              "boot_percentile", "boot_bca"),
) -> Dict[str, Dict[str, float]]:
    """Empirical coverage of nominal (1-alpha) intervals for a *known* accuracy.

    Exact shortcut used for the bootstrap arms
    ------------------------------------------
    For the mean of a binary vector containing k ones out of n, resampling n
    observations with replacement yields a replicate mean k*/n with
    k* ~ Binomial(n, k/n) *exactly*.  Drawing the binomial directly is
    therefore not an approximation of the bootstrap: it is the bootstrap,
    which reduces the cost of the study from O(n_mc B n) to O(n_mc B).

    The leave-one-out (jackknife) values of a binary mean take only two
    distinct values, so the BCa acceleration is likewise available in closed
    form.
    """
    rng = np.random.default_rng(seed)
    z_lo, z_hi = stats.norm.ppf(alpha / 2), stats.norm.ppf(1 - alpha / 2)
    hits = {m: 0 for m in methods}
    widths = {m: 0.0 for m in methods}

    ks = rng.binomial(n, p_true, size=n_mc)
    for k in ks:
        p_hat = k / n
        for m in methods:
            if m.startswith("boot_"):
                star = rng.binomial(n, p_hat, size=B) / n
                if m == "boot_percentile":
                    lo, hi = np.percentile(star, [100 * alpha / 2, 100 * (1 - alpha / 2)])
                else:  # boot_bca / boot_bca_midp
                    prop = np.mean(star < p_hat)
                    if m.endswith("midp"):
                        prop += 0.5 * np.mean(star == p_hat)
                    prop = min(max(prop, 0.5 / B), 1 - 0.5 / B)
                    z0 = stats.norm.ppf(prop)
                    # jackknife of a binary mean: k ones -> two distinct LOO values
                    jack = np.concatenate([
                        np.full(k, (k - 1) / (n - 1)),
                        np.full(n - k, k / (n - 1)),
                    ]) if n > 1 else np.array([p_hat])
                    d = jack.mean() - jack
                    den = 6.0 * (np.sum(d**2) ** 1.5)
                    a = float(np.sum(d**3) / den) if den > 0 else 0.0
                    a1 = stats.norm.cdf(z0 + (z0 + z_lo) / (1 - a * (z0 + z_lo)))
                    a2 = stats.norm.cdf(z0 + (z0 + z_hi) / (1 - a * (z0 + z_hi)))
                    if not np.isfinite(a1) or not np.isfinite(a2) or a1 >= a2:
                        lo, hi = np.percentile(star, [100 * alpha / 2, 100 * (1 - alpha / 2)])
                    else:
                        lo, hi = np.percentile(star, [100 * a1, 100 * a2])
            else:
                lo, hi = binomial_ci(int(k), n, alpha, method=m)
            hits[m] += int(lo <= p_true <= hi)
            widths[m] += hi - lo

    return {
        m: {
            "coverage": hits[m] / n_mc,
            "mean_width": widths[m] / n_mc,
            "mc_se": float(np.sqrt((hits[m] / n_mc) * (1 - hits[m] / n_mc) / n_mc)),
        }
        for m in methods
    }


Writing bootci.py


In [3]:
# =============================================================================
# 1b · Import the library and run its self-test against scikit-learn references
#      (a failed assertion here invalidates every number below — do not skip)
# =============================================================================
import importlib
import bootci as bc
importlib.reload(bc)
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             matthews_corrcoef, recall_score, precision_score,
                             balanced_accuracy_score, confusion_matrix,
                             classification_report)
from scipy import stats as sps

bc.set_global_seed(CONFIG["SEED"])
_rng = np.random.default_rng(0)

# (i) metric kernels must reproduce scikit-learn exactly
_n = 200
_yt = _rng.integers(0, 2, _n)
_ys = np.round(_rng.random(_n), 2)          # rounding forces genuine ties
_yp = (_ys > 0.5).astype(int)
_YT, _YP, _YS = _yt[None, :], _yp[None, :], _ys[None, :]
assert np.isclose(bc._accuracy(_YT, _YP, _YS)[0], accuracy_score(_yt, _yp))
assert np.isclose(bc._auroc(_YT, _YP, _YS)[0], roc_auc_score(_yt, _ys))
assert np.isclose(bc._f1(_YT, _YP, _YS)[0], f1_score(_yt, _yp))
assert np.isclose(bc._mcc(_YT, _YP, _YS)[0], matthews_corrcoef(_yt, _yp))
assert np.isclose(bc._balanced_accuracy(_YT, _YP, _YS)[0], balanced_accuracy_score(_yt, _yp))

# (ii) tie-aware AUROC must stay exact on resampled (duplicate-heavy) data
_idx = _rng.integers(0, _n, size=(30, _n))
_mine = bc._auroc(_yt[_idx], _yp[_idx], _ys[_idx])
_ref = np.array([roc_auc_score(_yt[i], _ys[i]) for i in _idx])
assert np.allclose(_mine, _ref), "AUROC tie correction is wrong"

# (iii) determinism
_a = bc.bootstrap_ci(_yt, _yp, _ys, metric="accuracy", B=1000, seed=7, return_replicates=False)
_b = bc.bootstrap_ci(_yt, _yp, _ys, metric="accuracy", B=1000, seed=7, return_replicates=False)
assert (_a.ci_low, _a.ci_high) == (_b.ci_low, _b.ci_high), "bootstrap is not reproducible"

# (iv) DeLong variance must agree with the bootstrap SE on a well-behaved case
_d = bc.delong_auc_ci(_yt, _ys)
_bs = bc.bootstrap_ci(_yt, _yp, _ys, metric="auroc", B=4000, method="percentile", seed=1)
assert abs(_d["se"] - _bs.se_boot) / _bs.se_boot < 0.15, "DeLong vs bootstrap SE mismatch"

print("bootci self-test: ALL CHECKS PASSED")
print(f"  sanity example -> accuracy {_a},  AUROC {_bs}")

bootci self-test: ALL CHECKS PASSED
  sanity example -> accuracy 0.5250 [0.4500, 0.5900],  AUROC 0.5340 [0.4549, 0.6159]


---
## Experiment 1 — Tabular baseline: Breast Cancer Wisconsin (Diagnostic)

569 samples, 30 features, 212 malignant. Labels are re-coded so that **malignant = 1** (the clinically positive class), because sensitivity/specificity are otherwise reported for the wrong class — a frequent silent error in papers using this dataset.

A stratified 70/30 split gives a held-out test set of **n = 171**, which is deliberately of the same order as the test sets used in many published clinical-AI studies.

In [4]:
# =============================================================================
# 2 · Experiment 1 — data, models, predictions
# =============================================================================
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

SEED = CONFIG["SEED"]
bc.set_global_seed(SEED)

X, y_raw = load_breast_cancer(return_X_y=True)
y = 1 - y_raw                      # scikit-learn: malignant=0, benign=1  ->  malignant=1
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=CONFIG["TAB_TEST_SIZE"], stratify=y, random_state=SEED)

print(f"total n = {X.shape[0]}, features = {X.shape[1]}, malignant prevalence = {y.mean():.4f}")
print(f"train n = {Xtr.shape[0]}  |  test n = {Xte.shape[0]}  "
      f"(malignant = {int(yte.sum())}, benign = {int((1-yte).sum())})")

TAB_MODELS = {
    "LogReg":   Pipeline([("sc", StandardScaler()),
                          ("clf", LogisticRegression(max_iter=5000, random_state=SEED))]),
    "SVM-RBF":  Pipeline([("sc", StandardScaler()),
                          ("clf", SVC(kernel="rbf", C=1.0, gamma="scale",
                                      probability=True, random_state=SEED))]),
    "RF":       RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "GaussNB":  Pipeline([("sc", StandardScaler()), ("clf", GaussianNB())]),
    "CART-d3":  DecisionTreeClassifier(max_depth=3, random_state=SEED),
}

TAB_PRED = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for name, model in TAB_MODELS.items():
    t0 = time.perf_counter()
    cv_acc = cross_val_score(model, Xtr, ytr, cv=cv, scoring="accuracy")
    model.fit(Xtr, ytr)
    TAB_PRED[name] = {
        "y_pred": model.predict(Xte),
        "y_score": model.predict_proba(Xte)[:, 1],
        "cv_mean": float(cv_acc.mean()),
        "cv_std": float(cv_acc.std(ddof=1)),
        "fit_s": time.perf_counter() - t0,
    }
    print(f"  {name:8s} 5-fold CV accuracy (train only) = "
          f"{cv_acc.mean():.4f} +/- {cv_acc.std(ddof=1):.4f}   "
          f"| test accuracy = {accuracy_score(yte, TAB_PRED[name]['y_pred']):.4f}")

total n = 569, features = 30, malignant prevalence = 0.3726
train n = 398  |  test n = 171  (malignant = 64, benign = 107)
  LogReg   5-fold CV accuracy (train only) = 0.9824 +/- 0.0069   | test accuracy = 0.9708
  SVM-RBF  5-fold CV accuracy (train only) = 0.9723 +/- 0.0209   | test accuracy = 0.9591
  RF       5-fold CV accuracy (train only) = 0.9472 +/- 0.0339   | test accuracy = 0.9766
  GaussNB  5-fold CV accuracy (train only) = 0.9295 +/- 0.0445   | test accuracy = 0.9357
  CART-d3  5-fold CV accuracy (train only) = 0.9221 +/- 0.0351   | test accuracy = 0.9181


In [5]:
# =============================================================================
# 3 · Experiment 1 — Table 1: point estimates with 95% BCa confidence intervals
# =============================================================================
TAB_METRICS = ["accuracy", "sensitivity", "specificity", "ppv", "f1", "mcc", "auroc"]
rows, boot_store = [], {}

with warnings.catch_warnings():
    warnings.simplefilter("always")
    for name, d in TAB_PRED.items():
        for metric in TAB_METRICS:
            r = bc.bootstrap_ci(
                yte, d["y_pred"], d["y_score"], metric=metric,
                B=CONFIG["B"], alpha=CONFIG["ALPHA"], method=CONFIG["CI_METHOD"],
                seed=SEED, stratified=CONFIG["STRATIFIED"], chunk_size=CONFIG["CHUNK"])
            boot_store[(name, metric)] = r.replicates
            rows.append({"model": name, **r.as_row()})

tab1 = pd.DataFrame(rows)
tab1["ci_str"] = tab1.apply(
    lambda r: f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1)

print(f"TABLE 1 — Breast Cancer Wisconsin, n_test = {len(yte)}, B = {CONFIG['B']}, "
      f"{int(100 * (1 - CONFIG['ALPHA']))}% BCa CI\n")
print(tab1.pivot(index="model", columns="metric", values="ci_str")[TAB_METRICS].to_string())

print("\nInterval WIDTH (the quantity the paper argues should be reported):\n")
print(tab1.pivot(index="model", columns="metric", values="width")[TAB_METRICS]
        .round(4).to_string())

degenerate = tab1[tab1.boundary_degenerate]
if len(degenerate):
    print("\n[!] BOUNDARY FAILURE — the non-parametric bootstrap returns a "
          "zero-width interval whenever the observed rate is exactly 0 or 1,\n"
          "    because every resample reproduces the same value. "
          "Closed-form Clopper-Pearson limits for those cells:")
    for _, r in degenerate.iterrows():
        cf = bc.rate_ci_closed_form(yte, TAB_PRED[r.model]["y_pred"], r.metric,
                                    alpha=CONFIG["ALPHA"], method="clopper_pearson")
        print(f"    {r.model:8s} {r.metric:12s} bootstrap = "
              f"[{r.ci_low:.3f}, {r.ci_high:.3f}]  ->  Clopper-Pearson = "
              f"[{cf['ci_low']:.3f}, {cf['ci_high']:.3f}]  (k={cf['k']}/{cf['n']})")

RESULTS["table1_tabular"] = tab1.drop(columns=["ci_str"]).to_dict(orient="records")
tab1.to_csv(OUT / "table1_tabular_metrics.csv", index=False)

/content/bootci.py:473: UserWarning: metric 'specificity' = 1.0000 lies on the boundary of the parameter space: the non-parametric bootstrap degenerates (zero-width interval). Use rate_ci_closed_form() / Clopper-Pearson.
  warnings.warn(
/content/bootci.py:473: UserWarning: metric 'ppv' = 1.0000 lies on the boundary of the parameter space: the non-parametric bootstrap degenerates (zero-width interval). Use rate_ci_closed_form() / Clopper-Pearson.
  warnings.warn(
/content/bootci.py:473: UserWarning: metric 'specificity' = 1.0000 lies on the boundary of the parameter space: the non-parametric bootstrap degenerates (zero-width interval). Use rate_ci_closed_form() / Clopper-Pearson.
  warnings.warn(
/content/bootci.py:473: UserWarning: metric 'ppv' = 1.0000 lies on the boundary of the parameter space: the non-parametric bootstrap degenerates (zero-width interval). Use rate_ci_closed_form() / Clopper-Pearson.
  warnings.warn(
/content/bootci.py:473: UserWarning: metric 'specificity' = 1.00

TABLE 1 — Breast Cancer Wisconsin, n_test = 171, B = 10000, 95% BCa CI

metric               accuracy           sensitivity           specificity                   ppv                    f1                   mcc                 auroc
model                                                                                                                                                            
CART-d3  0.918 [0.861, 0.947]  0.781 [0.667, 0.870]  1.000 [1.000, 1.000]  1.000 [1.000, 1.000]  0.877 [0.800, 0.930]  0.831 [0.743, 0.900]  0.919 [0.854, 0.959]
GaussNB  0.936 [0.883, 0.959]  0.875 [0.768, 0.939]  0.972 [0.922, 0.991]  0.949 [0.857, 0.985]  0.911 [0.841, 0.952]  0.862 [0.764, 0.925]  0.993 [0.981, 0.998]
LogReg   0.971 [0.924, 0.988]  0.938 [0.847, 0.984]  0.991 [0.946, 1.000]  0.984 [0.904, 1.000]  0.960 [0.907, 0.985]  0.938 [0.861, 0.976]  0.998 [0.988, 1.000]
RF       0.977 [0.936, 0.988]  0.938 [0.851, 0.984]  1.000 [1.000, 1.000]  1.000 [1.000, 1.000]  0.968 [0.919, 0.992] 

In [6]:
# =============================================================================
# 4 · Experiment 1 — Table 2: interval constructions for the SAME statistic
#     Wald is the negative control; Wilson, Agresti-Coull, Clopper-Pearson and
#     Jeffreys are the closed-form competitors; percentile and BCa are the
#     bootstrap arms.
# =============================================================================
rows = []
for name, d in TAB_PRED.items():
    k = int((d["y_pred"] == yte).sum()); n = len(yte)
    entry = {"model": name, "k": k, "n": n, "accuracy": k / n}
    for m in ["wald", "wilson", "agresti_coull", "clopper_pearson", "jeffreys"]:
        lo, hi = bc.binomial_ci(k, n, alpha=CONFIG["ALPHA"], method=m)
        entry[m] = f"[{lo:.4f}, {hi:.4f}]"
        entry[m + "_w"] = hi - lo
    for meth in ["percentile", "bca"]:
        r = bc.bootstrap_ci(yte, d["y_pred"], metric="accuracy", B=CONFIG["B"],
                            alpha=CONFIG["ALPHA"], method=meth, seed=SEED,
                            chunk_size=CONFIG["CHUNK"], return_replicates=False)
        entry["boot_" + meth] = f"[{r.ci_low:.4f}, {r.ci_high:.4f}]"
        entry["boot_" + meth + "_w"] = r.width
    rows.append(entry)

tab2 = pd.DataFrame(rows)
show = ["model", "accuracy", "wald", "wilson", "clopper_pearson",
        "boot_percentile", "boot_bca"]
print(f"TABLE 2 — 95% CI for ACCURACY under five constructions (n = {len(yte)})\n")
print(tab2[show].to_string(index=False))
print("\nInterval widths:")
print(tab2[["model", "wald_w", "wilson_w", "clopper_pearson_w",
            "boot_percentile_w", "boot_bca_w"]].round(4).to_string(index=False))

print("\nDiagnostic — does any interval leave the admissible range [0, 1]?")
for _, r in tab2.iterrows():
    hi_wald = float(r["wald"].split(",")[1].strip(" ]"))
    flag = "  <-- INADMISSIBLE (>1)" if hi_wald > 1 else ""
    print(f"  {r.model:8s} Wald upper limit = {hi_wald:.4f}{flag}")

RESULTS["table2_interval_methods"] = tab2.to_dict(orient="records")
tab2.to_csv(OUT / "table2_interval_methods.csv", index=False)

TABLE 2 — 95% CI for ACCURACY under five constructions (n = 171)

  model  accuracy             wald           wilson  clopper_pearson  boot_percentile         boot_bca
 LogReg  0.970760 [0.9455, 0.9960] [0.9334, 0.9874] [0.9331, 0.9904] [0.9415, 0.9942] [0.9240, 0.9883]
SVM-RBF  0.959064 [0.9294, 0.9888] [0.9179, 0.9800] [0.9175, 0.9834] [0.9298, 0.9883] [0.9123, 0.9766]
     RF  0.976608 [0.9540, 0.9993] [0.9414, 0.9909] [0.9412, 0.9936] [0.9532, 0.9942] [0.9357, 0.9883]
GaussNB  0.935673 [0.8989, 0.9724] [0.8885, 0.9637] [0.8878, 0.9675] [0.8947, 0.9708] [0.8830, 0.9591]
CART-d3  0.918129 [0.8770, 0.9592] [0.8673, 0.9506] [0.8664, 0.9545] [0.8772, 0.9591] [0.8611, 0.9474]

Interval widths:
  model  wald_w  wilson_w  clopper_pearson_w  boot_percentile_w  boot_bca_w
 LogReg  0.0505    0.0541             0.0574             0.0526      0.0643
SVM-RBF  0.0594    0.0621             0.0659             0.0585      0.0643
     RF  0.0453    0.0495             0.0524             0.0409      0

In [7]:
# =============================================================================
# 5 · Experiment 1 — paired model comparison
#     Two independent CIs that overlap do NOT imply "no significant difference".
#     The paired bootstrap reuses the same resampled patients for both models,
#     exploiting their correlation; exact McNemar is the closed-form check.
# =============================================================================
pairs = [("RF", "GaussNB"), ("RF", "CART-d3"), ("LogReg", "SVM-RBF")]
rows = []
for a, b in pairs:
    p_acc = bc.paired_bootstrap_diff(
        yte, TAB_PRED[a]["y_pred"], TAB_PRED[b]["y_pred"], metric="accuracy",
        B=CONFIG["B"], alpha=CONFIG["ALPHA"], seed=SEED, chunk_size=CONFIG["CHUNK"])
    p_auc = bc.paired_bootstrap_diff(
        yte, TAB_PRED[a]["y_pred"], TAB_PRED[b]["y_pred"],
        TAB_PRED[a]["y_score"], TAB_PRED[b]["y_score"], metric="auroc",
        B=CONFIG["B"], alpha=CONFIG["ALPHA"], seed=SEED, chunk_size=CONFIG["CHUNK"])
    mcn = bc.mcnemar_exact(yte, TAB_PRED[a]["y_pred"], TAB_PRED[b]["y_pred"])
    # naive, incorrect procedure: check whether two marginal CIs overlap
    ra = bc.bootstrap_ci(yte, TAB_PRED[a]["y_pred"], metric="accuracy", B=CONFIG["B"],
                         method="percentile", seed=SEED, return_replicates=False)
    rb = bc.bootstrap_ci(yte, TAB_PRED[b]["y_pred"], metric="accuracy", B=CONFIG["B"],
                         method="percentile", seed=SEED + 1, return_replicates=False)
    overlap = not (ra.ci_high < rb.ci_low or rb.ci_high < ra.ci_low)
    rows.append({
        "A": a, "B": b,
        "acc_A": p_acc["theta_a"], "acc_B": p_acc["theta_b"],
        "delta_acc": p_acc["delta"],
        "delta_ci": f"[{p_acc['ci_low']:.4f}, {p_acc['ci_high']:.4f}]",
        "boot_ASL_p": p_acc["asl_p"], "mcnemar_p": mcn["p_value"],
        "discordant_n01_n10": f"{mcn['n01']}/{mcn['n10']}",
        "delta_auc": p_auc["delta"],
        "delta_auc_ci": f"[{p_auc['ci_low']:.4f}, {p_auc['ci_high']:.4f}]",
        "paired_significant": p_acc["significant_at_alpha"],
        "marginal_CIs_overlap": overlap,
    })
tab3 = pd.DataFrame(rows)
print(f"TABLE 3 — paired comparison on the SAME test cohort (n = {len(yte)})\n")
print(tab3.to_string(index=False))
print("\nRead the last two columns together: a pair with overlapping marginal "
      "intervals that is nevertheless\nsignificant under the paired bootstrap is "
      "direct evidence that the 'overlap test' is invalid.")

# AUROC cross-check: bootstrap vs the asymptotic DeLong interval
print("\nAUROC — bootstrap percentile vs DeLong asymptotic interval:")
for name, d in TAB_PRED.items():
    rb_ = bc.bootstrap_ci(yte, d["y_pred"], d["y_score"], metric="auroc",
                          B=CONFIG["B"], method="percentile", seed=SEED,
                          return_replicates=False)
    dl = bc.delong_auc_ci(yte, d["y_score"], alpha=CONFIG["ALPHA"])
    flag = "  <-- DeLong upper limit exceeds 1" if dl["ci_high"] > 1 else ""
    print(f"  {name:8s} boot [{rb_.ci_low:.4f}, {rb_.ci_high:.4f}]   "
          f"DeLong [{dl['ci_low']:.4f}, {dl['ci_high']:.4f}]{flag}")

RESULTS["table3_paired"] = tab3.to_dict(orient="records")
tab3.to_csv(OUT / "table3_paired_comparison.csv", index=False)

TABLE 3 — paired comparison on the SAME test cohort (n = 171)

     A       B    acc_A    acc_B  delta_acc          delta_ci  boot_ASL_p  mcnemar_p discordant_n01_n10  delta_auc      delta_auc_ci  paired_significant  marginal_CIs_overlap
    RF GaussNB 0.976608 0.935673   0.040936  [0.0117, 0.0760]      0.0175   0.039062                8/1   0.003067 [-0.0014, 0.0092]                True                  True
    RF CART-d3 0.976608 0.918129   0.058480  [0.0292, 0.0994]      0.0028   0.001953               10/0   0.077249  [0.0326, 0.1307]                True                  True
LogReg SVM-RBF 0.970760 0.959064   0.011696 [-0.0175, 0.0409]      0.5324   0.687500                4/2   0.000292 [-0.0021, 0.0033]               False                  True

Read the last two columns together: a pair with overlapping marginal intervals that is nevertheless
significant under the paired bootstrap is direct evidence that the 'overlap test' is invalid.

AUROC — bootstrap percentile vs DeLong as

In [8]:
# =============================================================================
# 6 · Experiment 1 — identical point estimate, different uncertainty
#     Sub-cohorts are drawn WITHOUT replacement and stratified by class, so the
#     sub-cohort is a genuine smaller test set rather than a bootstrap sample.
# =============================================================================
def stratified_subsample(y, n_target, rng):
    """Class-proportional subsample without replacement; returns row indices."""
    if n_target is None or n_target >= len(y):
        return np.arange(len(y))
    idx = []
    classes, counts = np.unique(y, return_counts=True)
    quota = np.maximum(1, np.round(n_target * counts / counts.sum()).astype(int))
    for c, q in zip(classes, quota):
        pool = np.flatnonzero(y == c)
        idx.append(rng.choice(pool, size=min(q, pool.size), replace=False))
    return np.sort(np.concatenate(idx))

def sweep(y_true, y_pred, y_score, sizes, reps, metric="accuracy", seed=0,
          B=None, method="percentile", label=""):
    """CI width as a function of test-set size, averaged over `reps` sub-cohorts."""
    B = B or CONFIG["B"]
    rng = np.random.default_rng(seed)
    out = []
    for n_target in sizes:
        r_eff = 1 if (n_target is None or n_target >= len(y_true)) else reps
        rec = []
        for rep in range(r_eff):
            sel = stratified_subsample(y_true, n_target, rng)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                r = bc.bootstrap_ci(y_true[sel], y_pred[sel],
                                    None if y_score is None else y_score[sel],
                                    metric=metric, B=B, alpha=CONFIG["ALPHA"],
                                    method=method, seed=seed + 1000 * rep,
                                    chunk_size=CONFIG["CHUNK"], return_replicates=False)
            rec.append((len(sel), r.estimate, r.width, r.ci_low, r.ci_high))
        arr = np.array(rec, dtype=float)
        out.append({
            "label": label, "metric": metric, "n_requested": n_target or len(y_true),
            "n_actual": int(arr[0, 0]), "reps": r_eff,
            "est_mean": arr[:, 1].mean(), "est_sd": arr[:, 1].std(ddof=1) if r_eff > 1 else 0.0,
            "width_mean": arr[:, 2].mean(), "width_sd": arr[:, 2].std(ddof=1) if r_eff > 1 else 0.0,
            "width_min": arr[:, 2].min(), "width_max": arr[:, 2].max(),
        })
    return pd.DataFrame(out)

sizes_tab = [25, 50, 100, None]
sw_tab = pd.concat([
    sweep(yte, TAB_PRED["CART-d3"]["y_pred"], TAB_PRED["CART-d3"]["y_score"],
          sizes_tab, CONFIG["SWEEP_REPS"], "accuracy", seed=SEED, label="CART-d3 (acc~0.92)"),
    sweep(yte, TAB_PRED["RF"]["y_pred"], TAB_PRED["RF"]["y_score"],
          sizes_tab, CONFIG["SWEEP_REPS"], "accuracy", seed=SEED, label="RF (acc~0.98, near ceiling)"),
], ignore_index=True)

print("TABLE 4 — 95% CI width vs. test-set size (mean +/- SD over sub-cohorts)\n")
print(sw_tab[["label", "n_actual", "reps", "est_mean", "est_sd",
              "width_mean", "width_sd", "width_min", "width_max"]].round(4).to_string(index=False))
print("\nExpected behaviour: width ~ 1/sqrt(n) for a mid-range accuracy. "
      "For the near-ceiling model the\nrelationship breaks down — a small cohort "
      "in which the model happens to make no error yields a\nDECEPTIVELY NARROW "
      "interval. This ceiling artefact is itself a finding worth reporting.")

RESULTS["table4_sweep_tabular"] = sw_tab.to_dict(orient="records")
sw_tab.to_csv(OUT / "table4_sweep_tabular.csv", index=False)

TABLE 4 — 95% CI width vs. test-set size (mean +/- SD over sub-cohorts)

                      label  n_actual  reps  est_mean  est_sd  width_mean  width_sd  width_min  width_max
         CART-d3 (acc~0.92)        25    10    0.9320  0.0464      0.1560    0.0932     0.0000     0.2400
         CART-d3 (acc~0.92)        50    10    0.9220  0.0382      0.1400    0.0327     0.1000     0.1800
         CART-d3 (acc~0.92)       100    10    0.9190  0.0197      0.1060    0.0117     0.0900     0.1200
         CART-d3 (acc~0.92)       171     1    0.9181  0.0000      0.0819    0.0000     0.0819     0.0819
RF (acc~0.98, near ceiling)        25    10    0.9760  0.0337      0.0640    0.0868     0.0000     0.2000
RF (acc~0.98, near ceiling)        50    10    0.9800  0.0163      0.0540    0.0412     0.0000     0.1000
RF (acc~0.98, near ceiling)       100    10    0.9800  0.0125      0.0470    0.0241     0.0000     0.0700
RF (acc~0.98, near ceiling)       171     1    0.9766  0.0000      0.0409    0.

---
## Experiment 2 — Medical imaging: PneumoniaMNIST + CNN (PyTorch)

PneumoniaMNIST (MedMNIST v2) contains paediatric chest radiographs re-sampled to 28x28, with an official train/validation/test split. Two properties make it a good stress test for the paper's argument:

1. the test split is small (a few hundred images) — exactly the regime where point estimates mislead;
2. the classes are imbalanced, so accuracy alone hides the sensitivity/specificity trade-off.

Two operating points are reported: the default threshold 0.5 (primary) and a threshold tuned on the **validation** split by Youden's J and then frozen, so no test information leaks into either operating point.

In [9]:
# =============================================================================
# 7 · Experiment 2 — data loading (MedMNIST)
# =============================================================================
import medmnist
from medmnist import INFO
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms

FLAG = CONFIG["MED_FLAG"]
info = INFO[FLAG]
N_CH, N_CLS = info["n_channels"], len(info["label"])
assert info["task"] == "binary-class", (
    f"{FLAG} is '{info['task']}'. The binary metric kernels assume two classes; "
    "for multi-class flags compute one-vs-rest metrics instead.")
DataClass = getattr(medmnist, info["python_class"])
print(f"dataset : {FLAG}\ntask    : {info['task']}\nlabels  : {info['label']}")

norm = transforms.Normalize(mean=[0.5] * N_CH, std=[0.5] * N_CH)
# Mild geometric augmentation only. Horizontal flipping is deliberately avoided:
# it mirrors thoracic laterality and is not a label-preserving transform for
# chest radiographs.
tf_train = transforms.Compose([
    transforms.RandomAffine(degrees=7, translate=(0.06, 0.06), scale=(0.95, 1.05)),
    transforms.ToTensor(), norm])
tf_eval = transforms.Compose([transforms.ToTensor(), norm])

def make_split(split, tf):
    kw = dict(split=split, transform=tf, download=True, root=str(OUT / "medmnist"))
    try:
        return DataClass(size=CONFIG["IMG_SIZE"], **kw)     # medmnist >= 3.0
    except TypeError:
        return DataClass(**kw)                              # older API

(OUT / "medmnist").mkdir(parents=True, exist_ok=True)
ds_train = make_split("train", tf_train)
ds_val   = make_split("val",   tf_eval)
ds_test  = make_split("test",  tf_eval)

def prevalence(ds):
    lab = np.asarray(ds.labels).ravel()
    return len(lab), float((lab == 1).mean())

for nm, ds in [("train", ds_train), ("val", ds_val), ("test", ds_test)]:
    n_, p_ = prevalence(ds)
    print(f"  {nm:5s}: n = {n_:5d}   positive prevalence = {p_:.4f}")

def seed_worker(worker_id):
    ws = torch.initial_seed() % 2 ** 32
    np.random.seed(ws); random.seed(ws)

g = torch.Generator(); g.manual_seed(CONFIG["SEED"])
dl_kw = dict(num_workers=2, pin_memory=(DEVICE.type == "cuda"),
             worker_init_fn=seed_worker, generator=g, persistent_workers=True)
dl_train = DataLoader(ds_train, batch_size=CONFIG["BATCH"], shuffle=True,  drop_last=False, **dl_kw)
dl_val   = DataLoader(ds_val,   batch_size=256, shuffle=False, **dl_kw)
dl_test  = DataLoader(ds_test,  batch_size=256, shuffle=False, **dl_kw)

MED = {"n_test": len(ds_test), "n_train": len(ds_train), "n_val": len(ds_val),
       "flag": FLAG, "labels": info["label"]}

dataset : pneumoniamnist
task    : binary-class
labels  : {'0': 'normal', '1': 'pneumonia'}


100%|██████████| 4.17M/4.17M [00:06<00:00, 673kB/s]


  train: n =  4708   positive prevalence = 0.7421
  val  : n =   524   positive prevalence = 0.7424
  test : n =   624   positive prevalence = 0.6250


In [10]:
# =============================================================================
# 8 · Experiment 2 — CNN definition and training
#     ~0.3 M parameters: deliberately small, because the paper's claim concerns
#     the EVALUATION protocol, not architecture novelty.
# =============================================================================
class MedCNN(nn.Module):
    """Three-stage VGG-style CNN with BatchNorm and global average pooling.

    Forward complexity for a 28x28 input: O(B * 1.1e7) MACs.
    """
    def __init__(self, in_ch=1, n_classes=2, width=32, p_drop=0.3):
        super().__init__()
        def block(cin, cout, pool=True):
            layers = [nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                      nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                      nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                      nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
            if pool:
                layers.append(nn.MaxPool2d(2))
            return layers
        self.features = nn.Sequential(
            *block(in_ch,    width),          # 28 -> 14
            *block(width,    width * 2),      # 14 -> 7
            *block(width * 2, width * 4, pool=False),
            nn.AdaptiveAvgPool2d(1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(p_drop),
                                        nn.Linear(width * 4, n_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

bc.set_global_seed(CONFIG["SEED"])
model = MedCNN(in_ch=N_CH, n_classes=N_CLS, width=CONFIG["WIDTH"]).to(DEVICE)
n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_par:,}")

# Inverse-frequency class weights counteract the prevalence imbalance.
lab_tr = np.asarray(ds_train.labels).ravel()
cls_w = torch.tensor(len(lab_tr) / (N_CLS * np.bincount(lab_tr, minlength=N_CLS)),
                     dtype=torch.float32, device=DEVICE)
print(f"class weights: {cls_w.tolist()}")

criterion = nn.CrossEntropyLoss(weight=cls_w)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"],
                              weight_decay=CONFIG["WD"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["EPOCHS"])

@torch.no_grad()
def infer(net, loader):
    """Return (y_true, positive-class probability) for a whole split."""
    net.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        prob = torch.softmax(net(xb), dim=1)[:, 1]
        ys.append(yb.reshape(-1).cpu().numpy())
        ps.append(prob.cpu().numpy())
    return np.concatenate(ys).astype(int), np.concatenate(ps).astype(float)

history, best = [], {"auc": -1.0, "epoch": -1, "state": None}
t0 = time.perf_counter()
for epoch in range(1, CONFIG["EPOCHS"] + 1):
    model.train()
    run_loss = n_seen = 0
    for xb, yb in dl_train:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.reshape(-1).long().to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * xb.size(0); n_seen += xb.size(0)
    scheduler.step()
    yv, pv = infer(model, dl_val)
    v_auc = roc_auc_score(yv, pv); v_acc = accuracy_score(yv, (pv > 0.5).astype(int))
    history.append({"epoch": epoch, "train_loss": run_loss / n_seen,
                    "val_auc": v_auc, "val_acc": v_acc,
                    "lr": scheduler.get_last_lr()[0]})
    if v_auc > best["auc"]:
        best = {"auc": v_auc, "epoch": epoch,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
    print(f"  epoch {epoch:02d}/{CONFIG['EPOCHS']}  loss={run_loss/n_seen:.4f}  "
          f"val_acc={v_acc:.4f}  val_auc={v_auc:.4f}")

train_s = time.perf_counter() - t0
model.load_state_dict(best["state"])
print(f"\ntraining finished in {train_s:.1f}s — "
      f"best epoch {best['epoch']} (val AUROC = {best['auc']:.4f}) restored.")
MED.update(params=n_par, train_seconds=train_s, best_epoch=best["epoch"],
           best_val_auc=best["auc"], history=history)
torch.save(model.state_dict(), OUT / "cnn_best.pt")

trainable parameters: 287,138
class weights: [1.93904447555542, 0.6737263798713684]
  epoch 01/15  loss=0.2602  val_acc=0.9198  val_auc=0.9844
  epoch 02/15  loss=0.1471  val_acc=0.9389  val_auc=0.9930
  epoch 03/15  loss=0.1357  val_acc=0.9542  val_auc=0.9907
  epoch 04/15  loss=0.1238  val_acc=0.9141  val_auc=0.9958
  epoch 05/15  loss=0.1138  val_acc=0.9676  val_auc=0.9948
  epoch 06/15  loss=0.1121  val_acc=0.9733  val_auc=0.9977
  epoch 07/15  loss=0.0951  val_acc=0.9618  val_auc=0.9950
  epoch 08/15  loss=0.0918  val_acc=0.9542  val_auc=0.9974
  epoch 09/15  loss=0.0835  val_acc=0.9809  val_auc=0.9967
  epoch 10/15  loss=0.0787  val_acc=0.9332  val_auc=0.9970
  epoch 11/15  loss=0.0658  val_acc=0.9218  val_auc=0.9972
  epoch 12/15  loss=0.0689  val_acc=0.9714  val_auc=0.9969
  epoch 13/15  loss=0.0642  val_acc=0.9771  val_auc=0.9976
  epoch 14/15  loss=0.0590  val_acc=0.9714  val_auc=0.9975
  epoch 15/15  loss=0.0576  val_acc=0.9676  val_auc=0.9975

training finished in 383.9s — 

In [11]:
# =============================================================================
# 9 · Experiment 2 — frozen model, test-set inference, threshold from VALIDATION
# =============================================================================
y_val, p_val = infer(model, dl_val)
y_med, p_med = infer(model, dl_test)

# Youden's J on the validation split only -> no test-set leakage.
from sklearn.metrics import roc_curve
fpr, tpr, thr = roc_curve(y_val, p_val)
thr_star = float(thr[np.argmax(tpr - fpr)])
print(f"validation-optimal threshold (Youden J) = {thr_star:.4f}   "
      f"(default 0.5 kept as the primary operating point)")

pred_med_05 = (p_med >= 0.50).astype(int)
pred_med_J  = (p_med >= thr_star).astype(int)

print("\nconfusion matrix @0.50 (rows = truth, cols = prediction):")
print(confusion_matrix(y_med, pred_med_05))
print("\n", classification_report(y_med, pred_med_05,
                                  target_names=[info["label"]["0"], info["label"]["1"]],
                                  digits=4))

# Shallow reference model: multinomial logistic regression on raw pixels.
# It gives a genuinely different classifier for the paired-comparison analysis.
def flat_split(ds):
    """Raw (un-augmented) pixels flattened to a design matrix."""
    imgs = getattr(ds, "imgs", None)
    if imgs is None:                       # defensive: older/newer MedMNIST APIs
        imgs = np.stack([np.asarray(ds[i][0]).reshape(-1) for i in range(len(ds))])
        return imgs.astype(np.float32), np.asarray(ds.labels).ravel().astype(int)
    imgs = np.asarray(imgs, dtype=np.float32).reshape(len(imgs), -1) / 255.0
    return imgs, np.asarray(ds.labels).ravel().astype(int)

Xf_tr, yf_tr = flat_split(ds_train)
Xf_te, yf_te = flat_split(ds_test)
lr_pix = Pipeline([("sc", StandardScaler()),
                   ("clf", LogisticRegression(max_iter=2000, C=0.1,
                                              class_weight="balanced",
                                              random_state=SEED))]).fit(Xf_tr, yf_tr)
p_pix = lr_pix.predict_proba(Xf_te)[:, 1]
pred_pix = (p_pix >= 0.5).astype(int)
assert np.array_equal(yf_te, y_med), "test-set ordering differs between loaders"
print(f"\nCNN      test accuracy = {accuracy_score(y_med, pred_med_05):.4f}   "
      f"AUROC = {roc_auc_score(y_med, p_med):.4f}")
print(f"pixel-LR test accuracy = {accuracy_score(y_med, pred_pix):.4f}   "
      f"AUROC = {roc_auc_score(y_med, p_pix):.4f}")

MED_PRED = {
    "CNN@0.50":     {"y_pred": pred_med_05, "y_score": p_med},
    "CNN@Youden":   {"y_pred": pred_med_J,  "y_score": p_med},
    "PixelLogReg":  {"y_pred": pred_pix,    "y_score": p_pix},
}

validation-optimal threshold (Youden J) = 0.8042   (default 0.5 kept as the primary operating point)

confusion matrix @0.50 (rows = truth, cols = prediction):
[[127 107]
 [  0 390]]

               precision    recall  f1-score   support

      normal     1.0000    0.5427    0.7036       234
   pneumonia     0.7847    1.0000    0.8794       390

    accuracy                         0.8285       624
   macro avg     0.8924    0.7714    0.7915       624
weighted avg     0.8654    0.8285    0.8135       624


CNN      test accuracy = 0.8285   AUROC = 0.9749
pixel-LR test accuracy = 0.8734   AUROC = 0.9278


In [12]:
# =============================================================================
# 10 · Experiment 2 — Table 5: metrics with 95% CIs on the imaging test set
# =============================================================================
MED_METRICS = ["accuracy", "sensitivity", "specificity", "ppv", "npv",
               "f1", "balanced_accuracy", "mcc", "auroc"]
rows, med_boot = [], {}
with warnings.catch_warnings():
    warnings.simplefilter("always")
    for name, d in MED_PRED.items():
        for metric in MED_METRICS:
            r = bc.bootstrap_ci(y_med, d["y_pred"], d["y_score"], metric=metric,
                                B=CONFIG["B"], alpha=CONFIG["ALPHA"],
                                method=CONFIG["CI_METHOD"], seed=SEED,
                                stratified=CONFIG["STRATIFIED"], chunk_size=CONFIG["CHUNK"])
            med_boot[(name, metric)] = r.replicates
            rows.append({"model": name, **r.as_row()})

tab5 = pd.DataFrame(rows)
tab5["ci_str"] = tab5.apply(lambda r: f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1)
print(f"TABLE 5 — {FLAG}, n_test = {len(y_med)}, B = {CONFIG['B']}, 95% "
      f"{CONFIG['CI_METHOD'].upper()} CI\n")
print(tab5.pivot(index="model", columns="metric", values="ci_str")[MED_METRICS].to_string())
print("\nInterval width:")
print(tab5.pivot(index="model", columns="metric", values="width")[MED_METRICS].round(4).to_string())

# Closed-form cross-checks for every metric that is a simple rate.
print("\nCross-check — bootstrap vs exact Clopper-Pearson on the same rate:")
for metric in ["accuracy", "sensitivity", "specificity", "ppv", "npv"]:
    r = tab5[(tab5.model == "CNN@0.50") & (tab5.metric == metric)].iloc[0]
    cf = bc.rate_ci_closed_form(y_med, pred_med_05, metric, alpha=CONFIG["ALPHA"])
    print(f"  {metric:18s} boot [{r.ci_low:.4f}, {r.ci_high:.4f}]  "
          f"CP [{cf['ci_low']:.4f}, {cf['ci_high']:.4f}]  (k={cf['k']}/{cf['n']})")

dl_cnn = bc.delong_auc_ci(y_med, p_med, alpha=CONFIG["ALPHA"])
print(f"\nAUROC DeLong  [{dl_cnn['ci_low']:.4f}, {dl_cnn['ci_high']:.4f}]  "
      f"(SE = {dl_cnn['se']:.5f})")

# Paired comparison: CNN vs pixel-level logistic regression
p_cmp = bc.paired_bootstrap_diff(y_med, pred_med_05, pred_pix, p_med, p_pix,
                                 metric="auroc", B=CONFIG["B"], seed=SEED)
p_cmp_acc = bc.paired_bootstrap_diff(y_med, pred_med_05, pred_pix,
                                     metric="accuracy", B=CONFIG["B"], seed=SEED)
mcn_med = bc.mcnemar_exact(y_med, pred_med_05, pred_pix)
print(f"\nCNN vs PixelLogReg  delta_ACC = {p_cmp_acc['delta']:+.4f} "
      f"[{p_cmp_acc['ci_low']:.4f}, {p_cmp_acc['ci_high']:.4f}], ASL p = {p_cmp_acc['asl_p']:.4g}, "
      f"McNemar p = {mcn_med['p_value']:.4g}")
print(f"CNN vs PixelLogReg  delta_AUC = {p_cmp['delta']:+.4f} "
      f"[{p_cmp['ci_low']:.4f}, {p_cmp['ci_high']:.4f}], ASL p = {p_cmp['asl_p']:.4g}")

RESULTS["table5_imaging"] = tab5.drop(columns=["ci_str"]).to_dict(orient="records")
RESULTS["imaging_meta"] = MED
RESULTS["imaging_paired"] = {"acc": {k: v for k, v in p_cmp_acc.items() if k != "replicates"},
                             "auc": {k: v for k, v in p_cmp.items() if k != "replicates"},
                             "mcnemar": mcn_med, "delong_cnn": dl_cnn,
                             "threshold_youden": thr_star}
tab5.to_csv(OUT / "table5_imaging_metrics.csv", index=False)

/content/bootci.py:473: UserWarning: metric 'sensitivity' = 1.0000 lies on the boundary of the parameter space: the non-parametric bootstrap degenerates (zero-width interval). Use rate_ci_closed_form() / Clopper-Pearson.
  warnings.warn(
/content/bootci.py:473: UserWarning: metric 'npv' = 1.0000 lies on the boundary of the parameter space: the non-parametric bootstrap degenerates (zero-width interval). Use rate_ci_closed_form() / Clopper-Pearson.
  warnings.warn(


TABLE 5 — pneumoniamnist, n_test = 624, B = 10000, 95% BCA CI

metric                   accuracy           sensitivity           specificity                   ppv                   npv                    f1     balanced_accuracy                   mcc                 auroc
model                                                                                                                                                                                                            
CNN@0.50     0.829 [0.796, 0.856]  1.000 [1.000, 1.000]  0.543 [0.480, 0.607]  0.785 [0.747, 0.820]  1.000 [1.000, 1.000]  0.879 [0.855, 0.901]  0.771 [0.740, 0.804]  0.653 [0.604, 0.703]  0.975 [0.962, 0.983]
CNN@Youden   0.899 [0.872, 0.920]  0.987 [0.971, 0.995]  0.752 [0.695, 0.805]  0.869 [0.836, 0.899]  0.972 [0.939, 0.989]  0.924 [0.904, 0.942]  0.870 [0.840, 0.897]  0.789 [0.739, 0.833]  0.975 [0.962, 0.983]
PixelLogReg  0.873 [0.845, 0.896]  0.979 [0.961, 0.990]  0.697 [0.635, 0.753]  0.843 [0.808, 0.87

In [13]:
# =============================================================================
# 11 · Experiment 2 — Table 6: how the margin of error grows as the test cohort
#      shrinks (sub-cohorts drawn without replacement).
# =============================================================================
sizes = [s for s in CONFIG["SWEEP_SIZES"] if s is None or s < len(y_med)] + [None]
sizes = list(dict.fromkeys(sizes))          # de-duplicate, keep order

sw_med = pd.concat(
    [sweep(y_med, pred_med_05, p_med, sizes, CONFIG["SWEEP_REPS"], m,
           seed=SEED, method="percentile", label=f"CNN {m}")
     for m in ["accuracy", "sensitivity", "auroc"]],
    ignore_index=True)
sw_med["moe_mean"] = sw_med["width_mean"] / 2
sw_med["one_over_sqrt_n"] = 1 / np.sqrt(sw_med["n_actual"])

print(f"TABLE 6 — 95% CI width vs. test-cohort size ({FLAG}, B = {CONFIG['B']}, "
      f"{CONFIG['SWEEP_REPS']} independent sub-cohorts per size)\n")
print(sw_med[["label", "n_actual", "reps", "est_mean", "est_sd", "width_mean",
              "width_sd", "moe_mean"]].round(4).to_string(index=False))

# Empirical scaling exponent: log(width) = a + b log(n); theory predicts b = -0.5.
print("\nEmpirical scaling of the interval width (theoretical exponent = -0.5):")
for lab, grp in sw_med.groupby("label"):
    if len(grp) >= 3 and (grp.width_mean > 0).all():
        b, a = np.polyfit(np.log(grp.n_actual), np.log(grp.width_mean), 1)
        print(f"  {lab:18s} exponent = {b:+.3f}")

RESULTS["table6_sweep_imaging"] = sw_med.to_dict(orient="records")
sw_med.to_csv(OUT / "table6_sweep_imaging.csv", index=False)

TABLE 6 — 95% CI width vs. test-cohort size (pneumoniamnist, B = 10000, 10 independent sub-cohorts per size)

          label  n_actual  reps  est_mean  est_sd  width_mean  width_sd  moe_mean
   CNN accuracy        50    10    0.8360  0.0430      0.2000    0.0231    0.1000
   CNN accuracy       100    10    0.8550  0.0246      0.1380    0.0113    0.0690
   CNN accuracy       200    10    0.8290  0.0173      0.1040    0.0046    0.0520
   CNN accuracy       400    10    0.8282  0.0075      0.0740    0.0017    0.0370
   CNN accuracy       624     1    0.8285  0.0000      0.0593    0.0000    0.0296
CNN sensitivity        50    10    1.0000  0.0000      0.0000    0.0000    0.0000
CNN sensitivity       100    10    1.0000  0.0000      0.0000    0.0000    0.0000
CNN sensitivity       200    10    1.0000  0.0000      0.0000    0.0000    0.0000
CNN sensitivity       400    10    1.0000  0.0000      0.0000    0.0000    0.0000
CNN sensitivity       624     1    1.0000  0.0000      0.0000    0.000

---
## Experiment 3 — Monte-Carlo coverage study

A confidence interval is only worth reporting if it *covers*: an interval claiming 95% confidence must contain the true parameter in 95% of repeated experiments. This section measures that directly for a known ground-truth accuracy (paper, Table V).

**Exact shortcut.** For the mean of a binary vector with k ones out of n, a bootstrap replicate mean is distributed exactly as Binomial(n, k/n)/n. Sampling the binomial directly *is* the bootstrap, not an approximation of it, and cuts the cost of the study from O(M·B·n) to O(M·B).

For a simple proportion the bootstrap is not uniformly superior to the closed-form intervals; Section 16 examines metrics with no tractable closed form and paired comparisons.

In [14]:
# =============================================================================
# 12 · Experiment 3 — empirical coverage of nominal 95% intervals
# =============================================================================
COV_METHODS = ("wald", "wilson", "agresti_coull", "clopper_pearson",
               "jeffreys", "boot_percentile", "boot_bca", "boot_bca_midp")
rows = []
t0 = time.perf_counter()
for p_true in CONFIG["COV_P"]:
    for n in CONFIG["COV_N"]:
        cov = bc.coverage_simulation(p_true, n, n_mc=CONFIG["COV_MC"], B=CONFIG["COV_B"],
                                     alpha=CONFIG["ALPHA"], seed=int(1000 * p_true) + n,
                                     methods=COV_METHODS)
        for m, v in cov.items():
            rows.append({"p_true": p_true, "n": n, "method": m,
                         "coverage": v["coverage"], "mc_se": v["mc_se"],
                         "mean_width": v["mean_width"]})
        msg = ", ".join(m.replace("boot_", "B.") + "=" + format(v["coverage"], ".3f")
                        for m, v in cov.items())
        print(f"  p={p_true} n={n:4d} done ({msg})")
cov_df = pd.DataFrame(rows)
print(f"\ncoverage study finished in {time.perf_counter()-t0:.1f}s "
      f"(M = {CONFIG['COV_MC']} replications, B = {CONFIG['COV_B']})")

print("\nTABLE 7 — empirical coverage of nominal 95% intervals\n")
print(cov_df.pivot_table(index=["p_true", "n"], columns="method", values="coverage")
        .round(3).to_string())
print("\nMean interval width (efficiency — narrower is better AT EQUAL coverage):\n")
print(cov_df.pivot_table(index=["p_true", "n"], columns="method", values="mean_width")
        .round(4).to_string())

worst = cov_df.loc[cov_df.groupby("method").coverage.idxmin()]
print("\nWorst-case coverage per method across the grid:")
print(worst[["method", "p_true", "n", "coverage"]].sort_values("coverage").to_string(index=False))

RESULTS["table7_coverage"] = cov_df.to_dict(orient="records")
cov_df.to_csv(OUT / "table7_coverage.csv", index=False)

  p=0.8 n=  30 done (wald=0.948, wilson=0.969, agresti_coull=0.969, clopper_pearson=0.981, jeffreys=0.936, B.percentile=0.948, B.bca=0.970, B.bca_midp=0.981)
  p=0.8 n=  50 done (wald=0.925, wilson=0.948, agresti_coull=0.948, clopper_pearson=0.963, jeffreys=0.948, B.percentile=0.942, B.bca=0.967, B.bca_midp=0.964)
  p=0.8 n= 100 done (wald=0.933, wilson=0.941, agresti_coull=0.941, clopper_pearson=0.968, jeffreys=0.954, B.percentile=0.960, B.bca=0.958, B.bca_midp=0.963)
  p=0.8 n= 200 done (wald=0.939, wilson=0.958, agresti_coull=0.958, clopper_pearson=0.958, jeffreys=0.948, B.percentile=0.956, B.bca=0.956, B.bca_midp=0.959)
  p=0.8 n= 500 done (wald=0.946, wilson=0.950, agresti_coull=0.950, clopper_pearson=0.952, jeffreys=0.950, B.percentile=0.951, B.bca=0.953, B.bca_midp=0.951)
  p=0.9 n=  30 done (wald=0.817, wilson=0.979, agresti_coull=0.979, clopper_pearson=0.994, jeffreys=0.943, B.percentile=0.960, B.bca=0.942, B.bca_midp=0.958)
  p=0.9 n=  50 done (wald=0.872, wilson=0.971, agres

In [15]:
# =============================================================================
# 13 · Compute benchmark — is B = 10,000 actually expensive?
#      Supports the manuscript's "computational feasibility" argument.
# =============================================================================
def torch_bootstrap_accuracy(correct, B, seed, device, chunk=2000):
    """GPU-resident bootstrap of a mean. Time O(Bn), memory O(chunk*n)."""
    t = torch.as_tensor(np.asarray(correct, dtype=np.float32), device=device)
    n = t.numel()
    gen = torch.Generator(device=device); gen.manual_seed(seed)
    out = torch.empty(B, device=device)
    done = 0
    while done < B:
        b = min(chunk, B - done)
        idx = torch.randint(0, n, (b, n), generator=gen, device=device)
        out[done:done + b] = t[idx].mean(dim=1)
        done += b
    return out

correct = (pred_med_05 == y_med).astype(np.float64)
bench = []
for B_ in [1_000, 10_000, 100_000]:
    t0 = time.perf_counter()
    _ = bc.bootstrap_ci(y_med, pred_med_05, metric="accuracy", B=B_,
                        method="percentile", seed=0, chunk_size=5000,
                        return_replicates=False)
    t_np = time.perf_counter() - t0

    t0 = time.perf_counter()
    _ = bc.bootstrap_ci(y_med, pred_med_05, p_med, metric="auroc", B=B_,
                        method="percentile", seed=0, chunk_size=2000,
                        return_replicates=False)
    t_auc = time.perf_counter() - t0

    if DEVICE.type == "cuda":
        _ = torch_bootstrap_accuracy(correct, 256, 0, DEVICE)   # warm-up
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = torch_bootstrap_accuracy(correct, B_, 0, DEVICE, chunk=5000)
        torch.cuda.synchronize()
        t_gpu = time.perf_counter() - t0
    else:
        t_gpu = float("nan")
    bench.append({"B": B_, "n": len(y_med), "numpy_acc_s": t_np,
                  "numpy_auroc_s": t_auc, "torch_gpu_acc_s": t_gpu})

bench_df = pd.DataFrame(bench)
print(f"TABLE 8 — wall-clock cost of the bootstrap (n = {len(y_med)}, device = {DEVICE})\n")
print(bench_df.round(4).to_string(index=False))
print("\nInterpretation: at B = 10,000 the entire uncertainty analysis costs a "
      "fraction of a second —\nnegligible against the training time of "
      f"{MED['train_seconds']:.0f}s. Cost is not a defensible reason to omit CIs.")
RESULTS["table8_benchmark"] = bench_df.to_dict(orient="records")
bench_df.to_csv(OUT / "table8_benchmark.csv", index=False)

TABLE 8 — wall-clock cost of the bootstrap (n = 624, device = cpu)

     B   n  numpy_acc_s  numpy_auroc_s  torch_gpu_acc_s
  1000 624       0.0233         0.1081              NaN
 10000 624       0.1793         1.0697              NaN
100000 624       1.7131         7.5907              NaN

Interpretation: at B = 10,000 the entire uncertainty analysis costs a fraction of a second —
negligible against the training time of 384s. Cost is not a defensible reason to omit CIs.


In [16]:
# =============================================================================
# 14 · Figures (300 dpi, PNG + PDF, IEEE single-column geometry)
# =============================================================================
FIG = OUT / "figures"
def save(fig, name):
    fig.savefig(FIG / f"{name}.png"); fig.savefig(FIG / f"{name}.pdf")
    plt.close(fig); print(f"  saved figures/{name}.png|pdf")

# --- Fig 1: forest plot, tabular models -------------------------------------
sub = tab1[tab1.metric.isin(["accuracy", "sensitivity", "specificity", "auroc"])]
fig, axes = plt.subplots(1, 4, figsize=(9.5, 3.0), sharey=True)
for ax, metric in zip(axes, ["accuracy", "sensitivity", "specificity", "auroc"]):
    d = sub[sub.metric == metric].set_index("model").loc[list(TAB_MODELS)]
    ypos = np.arange(len(d))
    err = np.vstack([d.estimate - d.ci_low, d.ci_high - d.estimate])
    ax.errorbar(d.estimate, ypos, xerr=err, fmt="o", capsize=3, ms=4, lw=1.2, color="#1f3b73")
    ax.set_yticks(ypos); ax.set_yticklabels(d.index)
    ax.set_title(metric); ax.set_xlabel("estimate (95% CI)")
    ax.invert_yaxis()
fig.suptitle(f"Breast Cancer Wisconsin — 95% {CONFIG['CI_METHOD'].upper()} intervals "
             f"(n$_{{test}}$={len(yte)}, B={CONFIG['B']})", y=1.04)
save(fig, "fig1_forest_tabular")

# --- Fig 2: five interval constructions -------------------------------------
mdl = "GaussNB" if "GaussNB" in TAB_PRED else list(TAB_PRED)[0]
k = int((TAB_PRED[mdl]["y_pred"] == yte).sum()); n = len(yte)
methods = ["wald", "wilson", "agresti_coull", "clopper_pearson", "jeffreys"]
lo_hi = [bc.binomial_ci(k, n, method=m) for m in methods]
for meth in ["percentile", "bca"]:
    r = bc.bootstrap_ci(yte, TAB_PRED[mdl]["y_pred"], metric="accuracy", B=CONFIG["B"],
                        method=meth, seed=SEED, return_replicates=False)
    methods.append("bootstrap-" + meth); lo_hi.append((r.ci_low, r.ci_high))
fig, ax = plt.subplots(figsize=(4.2, 3.0))
for i, (m, (lo, hi)) in enumerate(zip(methods, lo_hi)):
    ax.plot([lo, hi], [i, i], lw=2.5, color="#1f3b73")
    ax.plot(k / n, i, "o", color="#c0392b", ms=4)
ax.axvline(1.0, ls=":", c="grey", lw=1)
ax.set_yticks(range(len(methods))); ax.set_yticklabels(methods)
ax.set_xlabel("accuracy"); ax.invert_yaxis()
ax.set_title(f"{mdl}: 95% CI, five constructions\n(k={k}/{n})")
save(fig, "fig2_interval_methods")

# --- Fig 3: bootstrap replicate distributions -------------------------------
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.0))
for ax, (store, ttl, mnames) in zip(
        axes,
        [(boot_store, f"Breast Cancer (n={len(yte)})", list(TAB_MODELS)),
         (med_boot, f"{FLAG} (n={len(y_med)})", list(MED_PRED))]):
    data = [store[(m, "accuracy")][np.isfinite(store[(m, "accuracy")])] for m in mnames]
    parts = ax.violinplot(data, showextrema=False, widths=0.85)
    for pc in parts["bodies"]:
        pc.set_facecolor("#4c72b0"); pc.set_alpha(0.6)
    for i, d in enumerate(data, start=1):
        lo, hi = np.percentile(d, [2.5, 97.5])
        ax.plot([i, i], [lo, hi], color="k", lw=1.5)
        ax.plot(i, np.mean(d), "o", color="#c0392b", ms=4)
    ax.set_xticks(range(1, len(mnames) + 1))
    ax.set_xticklabels(mnames, rotation=20, ha="right")
    ax.set_ylabel("bootstrap accuracy"); ax.set_title(ttl)
fig.suptitle("Bootstrap sampling distributions with 95% percentile limits", y=1.03)
save(fig, "fig3_bootstrap_distributions")

# --- Fig 4: CI width vs test-set size ---------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.0))
for lab, grp in sw_med.groupby("label"):
    axes[0].errorbar(grp.n_actual, grp.width_mean, yerr=grp.width_sd,
                     marker="o", capsize=3, ms=4, lw=1.2, label=lab)
ref_n = np.array(sorted(sw_med.n_actual.unique()), dtype=float)
base = sw_med[sw_med.label.str.contains("accuracy")].sort_values("n_actual")
if len(base):
    c = base.width_mean.iloc[0] * np.sqrt(base.n_actual.iloc[0])
    axes[0].plot(ref_n, c / np.sqrt(ref_n), "k--", lw=1, label=r"$\propto n^{-1/2}$")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("test-cohort size $n$"); axes[0].set_ylabel("95% CI width")
axes[0].set_title(f"{FLAG}"); axes[0].legend(fontsize=7)
for lab, grp in sw_tab.groupby("label"):
    axes[1].errorbar(grp.n_actual, grp.width_mean, yerr=grp.width_sd,
                     marker="s", capsize=3, ms=4, lw=1.2, label=lab)
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("test-cohort size $n$"); axes[1].set_ylabel("95% CI width")
axes[1].set_title("Breast Cancer Wisconsin"); axes[1].legend(fontsize=7)
fig.suptitle("Margin of error shrinks as $n^{-1/2}$ — except at the accuracy ceiling", y=1.04)
save(fig, "fig4_width_vs_n")

# --- Fig 5: coverage --------------------------------------------------------
plot_methods = ["wald", "wilson", "clopper_pearson", "boot_percentile", "boot_bca"]
ps = sorted(cov_df.p_true.unique())
fig, axes = plt.subplots(1, len(ps), figsize=(3.2 * len(ps), 3.0), sharey=True, squeeze=False)
for ax, p_true in zip(axes[0], ps):
    d = cov_df[cov_df.p_true == p_true]
    for m in plot_methods:
        dm = d[d.method == m].sort_values("n")
        ax.plot(dm.n, dm.coverage, marker="o", ms=3.5, lw=1.2, label=m)
    ax.axhline(1 - CONFIG["ALPHA"], color="k", ls="--", lw=1)
    ax.set_xscale("log"); ax.set_xlabel("$n$"); ax.set_title(f"true accuracy = {p_true}")
axes[0][0].set_ylabel("empirical coverage")
axes[0][-1].legend(fontsize=7, loc="lower right")
fig.suptitle(f"Coverage of nominal 95% intervals (M = {CONFIG['COV_MC']} replications)", y=1.04)
save(fig, "fig5_coverage")

# --- Fig 6: paired difference distribution ----------------------------------
fig, ax = plt.subplots(figsize=(4.2, 3.0))
d = p_cmp_acc["replicates"]
ax.hist(d, bins=60, color="#4c72b0", alpha=0.8, edgecolor="none")
ax.axvline(0, color="k", ls="--", lw=1.2, label="H$_0$: $\\Delta = 0$")
ax.axvline(p_cmp_acc["ci_low"], color="#c0392b", ls=":", lw=1.2)
ax.axvline(p_cmp_acc["ci_high"], color="#c0392b", ls=":", lw=1.2, label="95% CI")
ax.set_xlabel(r"$\Delta$ accuracy (CNN $-$ pixel logistic regression)")
ax.set_ylabel("bootstrap replicates")
ax.set_title(f"Paired bootstrap, n={len(y_med)}, B={CONFIG['B']}")
ax.legend(fontsize=7)
save(fig, "fig6_paired_delta")

# --- Fig 7: training curve --------------------------------------------------
h = pd.DataFrame(MED["history"])
fig, ax1 = plt.subplots(figsize=(4.2, 3.0))
ax1.plot(h.epoch, h.train_loss, color="#c0392b", marker="o", ms=3, label="train loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("training loss", color="#c0392b")
ax2 = ax1.twinx(); ax2.grid(False)
ax2.plot(h.epoch, h.val_auc, color="#1f3b73", marker="s", ms=3, label="val AUROC")
ax2.set_ylabel("validation AUROC", color="#1f3b73")
ax1.set_title(f"{FLAG} — CNN training ({MED['params']:,} parameters)")
save(fig, "fig7_training_curve")

  saved figures/fig1_forest_tabular.png|pdf
  saved figures/fig2_interval_methods.png|pdf
  saved figures/fig3_bootstrap_distributions.png|pdf
  saved figures/fig4_width_vs_n.png|pdf
  saved figures/fig5_coverage.png|pdf
  saved figures/fig6_paired_delta.png|pdf
  saved figures/fig7_training_curve.png|pdf


In [17]:
# =============================================================================
# 15 · Export — machine-readable results, LaTeX tables, BibTeX, ZIP archive
# =============================================================================
def df_to_latex(df, caption, label, floatfmt="%.4f"):
    body = df.to_latex(index=False, escape=True, float_format=lambda v: floatfmt % v)
    return ("\\begin{table}[!t]\n\\centering\n\\caption{%s}\n\\label{%s}\n%s\\end{table}\n"
            % (caption, label, body))

latex = "\n".join([
    df_to_latex(tab1.pivot(index="model", columns="metric", values="ci_str")[TAB_METRICS].reset_index(),
                "Breast Cancer Wisconsin: point estimates with 95\\% BCa confidence intervals "
                f"($n_{{test}}={len(yte)}$, $B={CONFIG['B']}$).", "tab:tabular"),
    df_to_latex(tab2[["model", "accuracy", "wald", "wilson", "clopper_pearson",
                      "boot_percentile", "boot_bca"]],
                "Five constructions of a 95\\% interval for accuracy.", "tab:intervals"),
    df_to_latex(tab5.pivot(index="model", columns="metric", values="ci_str")[MED_METRICS].reset_index(),
                f"{FLAG}: metrics with 95\\% confidence intervals "
                f"($n_{{test}}={len(y_med)}$).", "tab:imaging"),
    df_to_latex(sw_med[["label", "n_actual", "est_mean", "width_mean", "width_sd"]],
                "Interval width as a function of test-cohort size.", "tab:sweep"),
    df_to_latex(cov_df.pivot_table(index=["p_true", "n"], columns="method",
                                   values="coverage").reset_index(),
                "Empirical coverage of nominal 95\\% intervals.", "tab:coverage"),
])
(OUT / "tables_ieee.tex").write_text(latex, encoding="utf-8")

BIB = r"""
% ---------------------------------------------------------------------------
% Convenience BibTeX for the methods, datasets and software used here.
% The authoritative reference list is the one in the published paper.
% ---------------------------------------------------------------------------
@article{efron1979bootstrap,
  author  = {Efron, Bradley},
  title   = {Bootstrap Methods: Another Look at the Jackknife},
  journal = {The Annals of Statistics}, volume = {7}, number = {1},
  pages   = {1--26}, year = {1979}, doi = {10.1214/aos/1176344552}}

@article{efron1987better,
  author  = {Efron, Bradley},
  title   = {Better Bootstrap Confidence Intervals},
  journal = {Journal of the American Statistical Association},
  volume  = {82}, number = {397}, pages = {171--185}, year = {1987},
  doi     = {10.1080/01621459.1987.10478410}}

@book{efron1993introduction,
  author    = {Efron, Bradley and Tibshirani, Robert J.},
  title     = {An Introduction to the Bootstrap},
  publisher = {Chapman and Hall/CRC}, year = {1993},
  isbn      = {978-0412042317}}

@article{wilson1927probable,
  author  = {Wilson, Edwin B.},
  title   = {Probable Inference, the Law of Succession, and Statistical Inference},
  journal = {Journal of the American Statistical Association},
  volume  = {22}, number = {158}, pages = {209--212}, year = {1927},
  doi     = {10.1080/01621459.1927.10502953}}

@article{clopper1934use,
  author  = {Clopper, C. J. and Pearson, E. S.},
  title   = {The Use of Confidence or Fiducial Limits Illustrated in the Case
             of the Binomial}, journal = {Biometrika},
  volume  = {26}, number = {4}, pages = {404--413}, year = {1934},
  doi     = {10.1093/biomet/26.4.404}}

@article{brown2001interval,
  author  = {Brown, Lawrence D. and Cai, T. Tony and DasGupta, Anirban},
  title   = {Interval Estimation for a Binomial Proportion},
  journal = {Statistical Science}, volume = {16}, number = {2},
  pages   = {101--133}, year = {2001}, doi = {10.1214/ss/1009213286}}

@article{delong1988comparing,
  author  = {DeLong, Elizabeth R. and DeLong, David M. and
             Clarke-Pearson, Daniel L.},
  title   = {Comparing the Areas under Two or More Correlated Receiver
             Operating Characteristic Curves: A Nonparametric Approach},
  journal = {Biometrics}, volume = {44}, number = {3}, pages = {837--845},
  year    = {1988}, doi = {10.2307/2531595}}

@article{sun2014fast,
  author  = {Sun, Xu and Xu, Weichao},
  title   = {Fast Implementation of {DeLong}'s Algorithm for Comparing the
             Areas Under Correlated Receiver Operating Characteristic Curves},
  journal = {IEEE Signal Processing Letters}, volume = {21}, number = {11},
  pages   = {1389--1393}, year = {2014}, doi = {10.1109/LSP.2014.2337313}}

@article{yang2023medmnist,
  author  = {Yang, Jiancheng and Shi, Rui and Wei, Donglai and Liu, Zequan and
             Zhao, Lin and Ke, Bilian and Pfister, Hanspeter and Ni, Bingbing},
  title   = {{MedMNIST} v2 -- A Large-Scale Lightweight Benchmark for 2D and
             3D Biomedical Image Classification},
  journal = {Scientific Data}, volume = {10}, pages = {41}, year = {2023},
  doi     = {10.1038/s41597-022-01721-8}}

@inproceedings{yang2021medmnist,
  author    = {Yang, Jiancheng and Shi, Rui and Ni, Bingbing},
  title     = {{MedMNIST} Classification Decathlon: A Lightweight {AutoML}
               Benchmark for Medical Image Analysis},
  booktitle = {IEEE 18th International Symposium on Biomedical Imaging (ISBI)},
  pages     = {191--195}, year = {2021}, doi = {10.1109/ISBI48211.2021.9434062}}

@article{kermany2018identifying,
  author  = {Kermany, Daniel S. and others},
  title   = {Identifying Medical Diagnoses and Treatable Diseases by
             Image-Based Deep Learning},
  journal = {Cell}, volume = {172}, number = {5}, pages = {1122--1131},
  year    = {2018}, doi = {10.1016/j.cell.2018.02.010}}

@inproceedings{street1993nuclear,
  author    = {Street, W. Nick and Wolberg, William H. and
               Mangasarian, Olvi L.},
  title     = {Nuclear Feature Extraction for Breast Tumor Diagnosis},
  booktitle = {Biomedical Image Processing and Biomedical Visualization,
               Proc. SPIE 1905}, pages = {861--870}, year = {1993},
  doi       = {10.1117/12.148698}}

@article{varoquaux2022machine,
  author  = {Varoquaux, Ga{\"e}l and Cheplygina, Veronika},
  title   = {Machine Learning for Medical Imaging: Methodological Failures
             and Recommendations for the Future},
  journal = {npj Digital Medicine}, volume = {5}, pages = {48}, year = {2022},
  doi     = {10.1038/s41746-022-00592-y}}

@article{collins2024tripod,
  author  = {Collins, Gary S. and others},
  title   = {{TRIPOD+AI} Statement: Updated Guidance for Reporting Clinical
             Prediction Models That Use Regression or Machine Learning Methods},
  journal = {BMJ}, volume = {385}, pages = {e078378}, year = {2024},
  doi     = {10.1136/bmj-2023-078378}}

@article{harris2020array,
  author  = {Harris, Charles R. and others},
  title   = {Array Programming with {NumPy}},
  journal = {Nature}, volume = {585}, pages = {357--362}, year = {2020},
  doi     = {10.1038/s41586-020-2649-2}}

@article{virtanen2020scipy,
  author  = {Virtanen, Pauli and others},
  title   = {{SciPy} 1.0: Fundamental Algorithms for Scientific Computing
             in {Python}},
  journal = {Nature Methods}, volume = {17}, pages = {261--272}, year = {2020},
  doi     = {10.1038/s41592-019-0686-2}}

@article{pedregosa2011scikit,
  author  = {Pedregosa, Fabian and others},
  title   = {Scikit-learn: Machine Learning in {Python}},
  journal = {Journal of Machine Learning Research}, volume = {12},
  pages   = {2825--2830}, year = {2011}}

@inproceedings{paszke2019pytorch,
  author    = {Paszke, Adam and others},
  title     = {{PyTorch}: An Imperative Style, High-Performance Deep Learning
               Library},
  booktitle = {Advances in Neural Information Processing Systems 32},
  pages     = {8024--8035}, year = {2019}}
"""
(OUT / "references.bib").write_text(BIB, encoding="utf-8")

class NpEnc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.bool_,)):    return bool(o)
        if isinstance(o, np.ndarray):     return o.tolist()
        return super().default(o)

(OUT / "results_all.json").write_text(json.dumps(RESULTS, indent=2, cls=NpEnc), encoding="utf-8")

import shutil
shutil.rmtree(OUT / "medmnist", ignore_errors=True)      # keep the archive small
archive = shutil.make_archive("bootstrap_ci_results", "zip", root_dir=str(OUT))
print("\nfiles written:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size/1024:.1f} KB)")
print(f"\narchive: {archive}")

try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print(f"(automatic download unavailable: {e} — download '{archive}' from the file browser)")


files written:
  results/cnn_best.pt  (1137.5 KB)
  results/figures/fig1_forest_tabular.pdf  (26.1 KB)
  results/figures/fig1_forest_tabular.png  (94.6 KB)
  results/figures/fig2_interval_methods.pdf  (18.4 KB)
  results/figures/fig2_interval_methods.png  (91.4 KB)
  results/figures/fig3_bootstrap_distributions.pdf  (36.2 KB)
  results/figures/fig3_bootstrap_distributions.png  (235.7 KB)
  results/figures/fig4_width_vs_n.pdf  (30.9 KB)
  results/figures/fig4_width_vs_n.png  (152.3 KB)
  results/figures/fig5_coverage.pdf  (24.3 KB)
  results/figures/fig5_coverage.png  (188.2 KB)
  results/figures/fig6_paired_delta.pdf  (21.0 KB)
  results/figures/fig6_paired_delta.png  (74.1 KB)
  results/figures/fig7_training_curve.pdf  (17.3 KB)
  results/figures/fig7_training_curve.png  (147.8 KB)
  results/references.bib  (5.8 KB)
  results/results_all.json  (72.2 KB)
  results/table1_tabular_metrics.csv  (7.8 KB)
  results/table2_interval_methods.csv  (1.7 KB)
  results/table3_paired_comparison.cs

---
## Notes on interpretation

1. **Accuracy and other simple rates.** Table 7 shows that Wilson and Clopper–Pearson stay close to or above the nominal level, while Wald and both bootstrap intervals under-cover at small *n* and high accuracy. For a binomial proportion the score and exact intervals are preferable.
2. **Boundary collapse.** When an observed rate is exactly 0 or 1, every bootstrap variant returns a zero-width interval. The notebook flags these cells and reports the Clopper–Pearson interval next to them.
3. **Ceiling artefact.** For a near-perfect model a *small* cohort can yield a *narrower* bootstrap interval than a large one (Table 4), so interval width alone is not a quality signal.
4. **Cost.** Table 8 compares the wall-clock cost of B = 10,000 resamples with the training time of the CNN.

Section 16 extends the coverage study to F1, MCC, balanced accuracy and AUROC, and to the difference between two models evaluated on the same patients.

In [18]:
# =============================================================================
# 16 · Coverage for metrics without a closed-form interval, paired differences
#      and real-data paired comparisons (paper: Tables IV, VI, VII, VIII; Fig. 1)
# -----------------------------------------------------------------------------
# Part A  Empirical coverage of the percentile, BCa and bootstrap-SE normal
#         intervals (and DeLong for AUROC) for F1, MCC, balanced accuracy and
#         AUROC, whose sampling distributions have no standard closed form.
# Part B  Coverage of the interval for the DIFFERENCE between two correlated
#         models scored on the same patients (paired vs. unpaired analysis).
# Part C  Paired differences on the two real datasets, including F1, MCC and
#         balanced accuracy, for which no standard closed-form test exists.
#
# Ground truth. A binormal generative model gives EXACT population values for
# every metric, so "coverage" is measured against a known parameter rather than
# a plug-in estimate:
#     Y ~ Bernoulli(pi);  S | Y=1 ~ N(mu, s1^2);  S | Y=0 ~ N(0, 1)
#     AUROC = Phi( mu / sqrt(1 + s1^2) )                      (Mann-Whitney)
#     TPR   = 1 - Phi((t - mu)/s1),   FPR = 1 - Phi(t)
# and the confusion-matrix cells follow by mixing with the prevalence pi.
# Cohorts are drawn i.i.d. (n_pos random), which is the realistic clinical
# sampling scheme rather than a fixed case-control design.
#
# Cost: O(M * B * n log n), dominated by the AUROC midranks. All metrics share
# one set of resampling indices per replication, so adding F1/MCC to AUROC is
# nearly free. Measured wall-clock in the reported run (Colab CPU runtime):
# ~7.5 min (Part A) + ~2.5 min (Part B) + seconds (Part C).
# Set CONFIG["FAST_MODE"] = True beforehand for a ~1-minute smoke test.
#
# Run this cell AFTER the whole notebook: it reuses yte / TAB_PRED / y_med /
# MED_PRED and refreshes results_all.json and the ZIP archive at the end.
# =============================================================================
import numpy as np, pandas as pd, warnings, time, json
from scipy import stats as sps
import bootci as bc

ADDON = {
    "M":        1000,        # Monte-Carlo replications per configuration
    "B":         999,        # bootstrap resamples inside each replication
    "N_GRID":   [50, 100, 200, 500],
    "N_GRID_PAIRED": [100, 300],
    "M_PAIRED": 600,
    "ALPHA":    CONFIG["ALPHA"] if "CONFIG" in globals() else 0.05,
    "CHUNK":    1000,
    "SEED":     20240,
    "METRICS":  ["accuracy", "f1", "mcc", "balanced_accuracy", "auroc"],
}
if "CONFIG" in globals() and CONFIG.get("FAST_MODE"):
    ADDON.update(M=150, B=400, N_GRID=[100, 500], N_GRID_PAIRED=[200], M_PAIRED=150)

ALPHA = ADDON["ALPHA"]
ZCRIT = sps.norm.ppf(1 - ALPHA / 2)
Phi = sps.norm.cdf


# ---------------------------------------------------------------- ground truth
def true_metrics(pi: float, mu: float, s1: float, t: float) -> dict:
    """Exact population value of every metric under the binormal model."""
    tpr = 1.0 - Phi((t - mu) / s1)
    fpr = 1.0 - Phi(t)
    tp, fn = pi * tpr, pi * (1 - tpr)
    fp, tn = (1 - pi) * fpr, (1 - pi) * (1 - fpr)
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return {
        "accuracy":          tp + tn,
        "sensitivity":       tpr,
        "specificity":       1 - fpr,
        "ppv":               tp / (tp + fp),
        "npv":               tn / (tn + fn),
        "f1":                2 * tp / (2 * tp + fp + fn),
        "balanced_accuracy": 0.5 * (tpr + 1 - fpr),
        "mcc":               (tp * tn - fp * fn) / den,
        "auroc":             Phi(mu / np.sqrt(1.0 + s1 ** 2)),
    }


def draw_cohort(n, pi, mu, s1, t, rng):
    y = rng.binomial(1, pi, n)
    s = np.where(y == 1, rng.normal(mu, s1, n), rng.normal(0.0, 1.0, n))
    return y, s, (s >= t).astype(np.int64)


# ------------------------------------------- one bootstrap pass, many metrics
def boot_replicates_multi(y, score, pred, metrics, B, rng, chunk):
    """B bootstrap replicates of several metrics sharing ONE index stream."""
    n = y.size
    out = {m: np.empty(B, dtype=np.float64) for m in metrics}
    p = 0
    for idx in bc._iter_index_chunks(n, B, rng, None, chunk):
        b = idx.shape[0]
        yt, yp, ys = y[idx], pred[idx], score[idx]
        for m in metrics:
            out[m][p:p + b] = bc.METRICS[m](yt, yp, ys)
        p += b
    return out


def jackknife_multi(y, score, pred, metrics):
    """Leave-one-out replicates of several metrics sharing ONE index matrix."""
    n = y.size
    loo = np.stack([np.delete(np.arange(n), i) for i in range(n)])
    yt, yp, ys = y[loo], pred[loo], score[loo]
    return {m: bc.METRICS[m](yt, yp, ys) for m in metrics}


def intervals_from(theta_hat, star, jack, alpha):
    """Percentile, BCa and bootstrap-SE normal intervals for one metric."""
    ok = np.isfinite(star)
    st = star[ok]
    if st.size < 10:
        return {k: (np.nan, np.nan) for k in ("percentile", "bca", "normal")}
    lo_p, hi_p = np.percentile(st, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    lo_b, hi_b, _, _ = bc._bca_ci(theta_hat, st, jack, alpha)
    se = st.std(ddof=1)
    return {"percentile": (lo_p, hi_p), "bca": (lo_b, hi_b),
            "normal": (theta_hat - ZCRIT * se, theta_hat + ZCRIT * se)}


# =============================================================================
# PART A — coverage for metrics with no closed-form interval
# =============================================================================
SETTINGS = [
    # label,                pi,   mu,  s1,   t
    ("moderate (AUROC~0.85)", 0.40, 1.50, 1.00, 0.75),
    ("high (AUROC~0.97)",     0.40, 2.70, 1.00, 1.35),
]
ARMS = ["percentile", "bca", "normal"]
RANGE = {"accuracy": (0, 1), "f1": (0, 1), "balanced_accuracy": (0, 1),
         "auroc": (0, 1), "mcc": (-1, 1)}

rows, t_start = [], time.perf_counter()
for label, pi, mu, s1, t in SETTINGS:
    truth = true_metrics(pi, mu, s1, t)
    print(f"\nsetting: {label}  |  true "
          + ", ".join(f"{m}={truth[m]:.4f}" for m in ADDON["METRICS"]))
    for n in ADDON["N_GRID"]:
        rng = np.random.default_rng(ADDON["SEED"] + n + int(1000 * mu))
        hit = {(m, a): 0 for m in ADDON["METRICS"] for a in ARMS}
        wid = {(m, a): 0.0 for m in ADDON["METRICS"] for a in ARMS}
        bad = {(m, a): 0 for m in ADDON["METRICS"] for a in ARMS}   # outside range
        bias = {m: 0.0 for m in ADDON["METRICS"]}
        dl_hit = dl_wid = 0
        n_used = 0
        t0 = time.perf_counter()
        for _ in range(ADDON["M"]):
            y, s, pred = draw_cohort(n, pi, mu, s1, t, rng)
            if y.sum() < 2 or (1 - y).sum() < 2:      # degenerate cohort
                continue
            n_used += 1
            hat = {m: float(bc.METRICS[m](y[None, :], pred[None, :], s[None, :])[0])
                   for m in ADDON["METRICS"]}
            star = boot_replicates_multi(y, s, pred, ADDON["METRICS"],
                                         ADDON["B"], rng, ADDON["CHUNK"])
            jack = jackknife_multi(y, s, pred, ADDON["METRICS"])
            for m in ADDON["METRICS"]:
                bias[m] += hat[m] - truth[m]
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    ivs = intervals_from(hat[m], star[m], jack[m], ALPHA)
                lo_r, hi_r = RANGE[m]
                for a in ARMS:
                    lo, hi = ivs[a]
                    if not np.isfinite(lo):
                        continue
                    hit[(m, a)] += int(lo <= truth[m] <= hi)
                    wid[(m, a)] += hi - lo
                    bad[(m, a)] += int(lo < lo_r - 1e-12 or hi > hi_r + 1e-12)
            dl = bc.delong_auc_ci(y, s, alpha=ALPHA)
            dl_hit += int(dl["ci_low"] <= truth["auroc"] <= dl["ci_high"])
            dl_wid += dl["ci_high"] - dl["ci_low"]

        for m in ADDON["METRICS"]:
            for a in ARMS:
                c = hit[(m, a)] / n_used
                rows.append({"setting": label, "n": n, "metric": m, "method": a,
                             "true_value": truth[m], "coverage": c,
                             "mc_se": np.sqrt(c * (1 - c) / n_used),
                             "mean_width": wid[(m, a)] / n_used,
                             "inadmissible_rate": bad[(m, a)] / n_used,
                             "mean_bias": bias[m] / n_used, "M_used": n_used})
        c = dl_hit / n_used
        rows.append({"setting": label, "n": n, "metric": "auroc", "method": "delong",
                     "true_value": truth["auroc"], "coverage": c,
                     "mc_se": np.sqrt(c * (1 - c) / n_used),
                     "mean_width": dl_wid / n_used, "inadmissible_rate": np.nan,
                     "mean_bias": bias["auroc"] / n_used, "M_used": n_used})
        print(f"  n={n:4d} ({n_used} usable cohorts, {time.perf_counter()-t0:.1f}s): "
              + "  ".join(f"{m}/bca={hit[(m,'bca')]/n_used:.3f}"
                          for m in ["f1", "mcc", "auroc"]))

cov_cplx = pd.DataFrame(rows)
print(f"\nPart A finished in {time.perf_counter()-t_start:.0f}s")

print("\nTABLE 9 — empirical coverage of nominal "
      f"{int(100*(1-ALPHA))}% intervals for metrics WITHOUT a closed form\n")
print(cov_cplx.pivot_table(index=["setting", "metric", "n"], columns="method",
                           values="coverage").round(3).to_string())
print("\nMean width (compare only at comparable coverage):\n")
print(cov_cplx.pivot_table(index=["setting", "metric", "n"], columns="method",
                           values="mean_width").round(4).to_string())
print("\nShare of intervals falling outside the metric's admissible range:\n")
print(cov_cplx.pivot_table(index=["setting", "metric"], columns="method",
                           values="inadmissible_rate").round(4).to_string())
print("\nFinite-sample bias of the point estimate (nonlinear metrics are NOT unbiased):\n")
print(cov_cplx[cov_cplx.method == "bca"]
      .pivot_table(index=["setting", "metric"], columns="n", values="mean_bias")
      .round(4).to_string())

worst = (cov_cplx.loc[cov_cplx.groupby(["metric", "method"]).coverage.idxmin()]
         .sort_values("coverage")[["metric", "method", "setting", "n", "coverage"]])
print("\nWorst-case coverage per (metric, method):")
print(worst.to_string(index=False))

RESULTS["table9_coverage_complex"] = cov_cplx.to_dict(orient="records")
cov_cplx.to_csv(OUT / "table9_coverage_complex.csv", index=False)


# =============================================================================
# PART B — coverage for the PAIRED DIFFERENCE between two correlated models
#   Model A and model B score the same patient; their errors are correlated
#   (rho), which is exactly why an unpaired analysis is wrong.
# =============================================================================
def draw_pair(n, pi, mu_a, mu_b, rho, t_a, t_b, rng):
    y = rng.binomial(1, pi, n)
    z1, z2 = rng.normal(size=n), rng.normal(size=n)
    s_a = mu_a * y + z1
    s_b = mu_b * y + rho * z1 + np.sqrt(1 - rho ** 2) * z2
    return y, s_a, s_b, (s_a >= t_a).astype(np.int64), (s_b >= t_b).astype(np.int64)


PAIR_SETTINGS = [
    # label,                     pi,  mu_a, mu_b, rho,  t_a,  t_b
    ("small gap, rho=0.8",       0.40, 1.80, 1.50, 0.80, 0.90, 0.75),
    ("null (A == B), rho=0.8",   0.40, 1.60, 1.60, 0.80, 0.80, 0.80),
]
PAIR_METRICS = ["accuracy", "f1", "mcc", "auroc"]

rows, t_start = [], time.perf_counter()
for label, pi, mu_a, mu_b, rho, t_a, t_b in PAIR_SETTINGS:
    ta, tb = true_metrics(pi, mu_a, 1.0, t_a), true_metrics(pi, mu_b, 1.0, t_b)
    d_true = {m: ta[m] - tb[m] for m in PAIR_METRICS}
    print(f"\npaired setting: {label}  |  true deltas "
          + ", ".join(f"{m}={d_true[m]:+.4f}" for m in PAIR_METRICS))
    for n in ADDON["N_GRID_PAIRED"]:
        rng = np.random.default_rng(ADDON["SEED"] + 7 * n + int(100 * mu_a))
        hit = {m: 0 for m in PAIR_METRICS}
        wid = {m: 0.0 for m in PAIR_METRICS}
        rej = {m: 0 for m in PAIR_METRICS}          # ASL < alpha
        hit_u = {m: 0 for m in PAIR_METRICS}        # unpaired (independent-CI) arm
        wid_u = {m: 0.0 for m in PAIR_METRICS}
        corr = {m: 0.0 for m in PAIR_METRICS}       # corr(theta*_A, theta*_B)
        n_used = 0
        t0 = time.perf_counter()
        for _ in range(ADDON["M_PAIRED"]):
            y, sa, sb, pa, pb = draw_pair(n, pi, mu_a, mu_b, rho, t_a, t_b, rng)
            if y.sum() < 2 or (1 - y).sum() < 2:
                continue
            n_used += 1
            # ONE index stream, applied to both models -> the paired replicates,
            # and the two marginal replicate sets come out of the same pass.
            rgp = np.random.default_rng(int(rng.integers(1 << 31)))
            star_a = {m: np.empty(ADDON["B"]) for m in PAIR_METRICS}
            star_b = {m: np.empty(ADDON["B"]) for m in PAIR_METRICS}
            dstar = {m: np.empty(ADDON["B"]) for m in PAIR_METRICS}
            p = 0
            for idx in bc._iter_index_chunks(n, ADDON["B"], rgp, None, ADDON["CHUNK"]):
                b = idx.shape[0]
                yt = y[idx]
                for m in PAIR_METRICS:
                    va = bc.METRICS[m](yt, pa[idx], sa[idx])
                    vb = bc.METRICS[m](yt, pb[idx], sb[idx])
                    star_a[m][p:p + b], star_b[m][p:p + b] = va, vb
                    dstar[m][p:p + b] = va - vb
                p += b
            for m in PAIR_METRICS:
                d_hat = (float(bc.METRICS[m](y[None, :], pa[None, :], sa[None, :])[0])
                         - float(bc.METRICS[m](y[None, :], pb[None, :], sb[None, :])[0]))
                dv = dstar[m][np.isfinite(dstar[m])]
                lo, hi = np.percentile(dv, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
                hit[m] += int(lo <= d_true[m] <= hi)
                wid[m] += hi - lo
                asl = float(np.mean(np.abs(dv - d_hat) >= abs(d_hat)))
                rej[m] += int(asl < ALPHA)
                # Straw man: treat the two models as independent and add variances.
                ma, mb = star_a[m], star_b[m]
                good = np.isfinite(ma) & np.isfinite(mb)
                sa_, sb_ = ma[good].std(ddof=1), mb[good].std(ddof=1)
                se_u = np.sqrt(sa_ ** 2 + sb_ ** 2)
                hit_u[m] += int(d_hat - ZCRIT * se_u <= d_true[m] <= d_hat + ZCRIT * se_u)
                wid_u[m] += 2 * ZCRIT * se_u
                if sa_ > 0 and sb_ > 0:
                    corr[m] += float(np.corrcoef(ma[good], mb[good])[0, 1])
        for m in PAIR_METRICS:
            c, cu = hit[m] / n_used, hit_u[m] / n_used
            rows.append({"setting": label, "n": n, "metric": m,
                         "true_delta": d_true[m],
                         "coverage_paired": c, "coverage_unpaired": cu,
                         "mc_se": np.sqrt(c * (1 - c) / n_used),
                         "mean_width_paired": wid[m] / n_used,
                         "mean_width_unpaired": wid_u[m] / n_used,
                         "width_ratio": (wid_u[m] / max(wid[m], 1e-12)),
                         "boot_corr_AB": corr[m] / n_used,
                         "rejection_rate": rej[m] / n_used, "M_used": n_used})
        print(f"  n={n:4d} ({time.perf_counter()-t0:.1f}s): "
              + "  ".join(f"{m}: paired={hit[m]/n_used:.3f} / unpaired={hit_u[m]/n_used:.3f}"
                          for m in ["f1", "mcc"]))

pair_cov = pd.DataFrame(rows)
print(f"\nPart B finished in {time.perf_counter()-t_start:.0f}s")
print("\nTABLE 10 — coverage of the interval for a DIFFERENCE between two "
      "correlated models\n")
print(pair_cov.pivot_table(index=["setting", "metric"], columns="n",
                           values=["coverage_paired", "coverage_unpaired"])
      .round(3).to_string())
print("\nRejection rate at alpha = %.2f (for the null setting this IS the type-I "
      "error rate;\nfor the non-null setting it is statistical power):\n" % ALPHA)
print(pair_cov.pivot_table(index=["setting", "metric"], columns="n",
                           values="rejection_rate").round(3).to_string())
print("\nWhy pairing matters — correlation between the two models' bootstrap "
      "replicates, and how\nmuch wider the interval becomes if that correlation "
      "is ignored:\n")
print(pair_cov.pivot_table(index=["setting", "metric"], columns="n",
                           values=["boot_corr_AB", "width_ratio"]).round(3).to_string())

RESULTS["table10_paired_coverage"] = pair_cov.to_dict(orient="records")
pair_cov.to_csv(OUT / "table10_paired_coverage.csv", index=False)


# =============================================================================
# PART C — real-data paired comparisons on F1, MCC and balanced accuracy,
#          i.e. the metrics for which no standard closed-form test exists.
# =============================================================================
def format_p(p, B):
    """A bootstrap ASL of exactly 0 means 'below resolution', never 'zero'."""
    return f"<{1.0/B:.0e}" if p <= 0 else f"{p:.4g}"

CMP_METRICS = ["accuracy", "f1", "mcc", "balanced_accuracy", "auroc"]
CLOSED_FORM = {"accuracy": "McNemar (exact)", "auroc": "DeLong",
               "f1": "none", "mcc": "none", "balanced_accuracy": "none"}

jobs = []
if "TAB_PRED" in globals():
    jobs += [("BreastCancer", yte, "RF", "GaussNB", TAB_PRED),
             ("BreastCancer", yte, "RF", "CART-d3", TAB_PRED),
             ("BreastCancer", yte, "LogReg", "SVM-RBF", TAB_PRED)]
if "MED_PRED" in globals():
    jobs += [(FLAG, y_med, "CNN@0.50", "PixelLogReg", MED_PRED)]

rows = []
for ds, yt_, a, b, store in jobs:
    for m in CMP_METRICS:
        r = bc.paired_bootstrap_diff(
            yt_, store[a]["y_pred"], store[b]["y_pred"],
            store[a]["y_score"], store[b]["y_score"], metric=m,
            B=CONFIG["B"], alpha=ALPHA, seed=CONFIG["SEED"], chunk_size=CONFIG["CHUNK"])
        rows.append({
            "dataset": ds, "A": a, "B": b, "metric": m,
            "theta_A": r["theta_a"], "theta_B": r["theta_b"], "delta": r["delta"],
            "ci": f"[{r['ci_low']:.4f}, {r['ci_high']:.4f}]",
            "ci_low": r["ci_low"], "ci_high": r["ci_high"],
            "width": r["ci_high"] - r["ci_low"],
            "ASL_p": format_p(r["asl_p"], CONFIG["B"]),
            "significant": r["significant_at_alpha"],
            "closed_form_alternative": CLOSED_FORM[m],
        })
tab11 = pd.DataFrame(rows)
print("\nTABLE 11 — paired differences on metrics with NO closed-form interval "
      f"(B = {CONFIG['B']})\n")
print(tab11[["dataset", "A", "B", "metric", "delta", "ci", "ASL_p",
             "significant", "closed_form_alternative"]].to_string(index=False))
print("\nRows whose last column reads 'none' have no standard closed-form "
      "alternative;\nthe paired bootstrap is the only interval reported for them.")

# Corrected reporting of the ASL values that were printed as exactly 0 earlier.
if "p_cmp_acc" in globals():
    print(f"\nCorrected ASL reporting for Experiment 2 (B = {CONFIG['B']}):")
    print(f"  delta accuracy = {p_cmp_acc['delta']:+.4f} "
          f"[{p_cmp_acc['ci_low']:.4f}, {p_cmp_acc['ci_high']:.4f}], "
          f"ASL p = {format_p(p_cmp_acc['asl_p'], CONFIG['B'])}")
    print(f"  delta AUROC    = {p_cmp['delta']:+.4f} "
          f"[{p_cmp['ci_low']:.4f}, {p_cmp['ci_high']:.4f}], "
          f"ASL p = {format_p(p_cmp['asl_p'], CONFIG['B'])}")

# Multiplicity: several metrics on one cohort are highly dependent tests.
from itertools import groupby
print("\nMultiplicity note — Holm-adjusted ASL within each model pair:")
for (ds, a, b), grp in tab11.groupby(["dataset", "A", "B"], sort=False):
    praw = np.array([1.0 / CONFIG["B"] if p.startswith("<") else float(p)
                     for p in grp.ASL_p])
    order = np.argsort(praw)
    adj = np.empty_like(praw)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(praw) - rank) * praw[i])
        adj[i] = min(1.0, running)
    print(f"  {ds} {a} vs {b}: " + ", ".join(
        f"{m}={p:.3g}" for m, p in zip(grp.metric, adj)))

RESULTS["table11_paired_complex"] = tab11.to_dict(orient="records")
tab11.to_csv(OUT / "table11_paired_complex.csv", index=False)


# =============================================================================
# FIGURE 8 — coverage for metrics without a closed form
#            (Fig. 1 of the paper is redrawn from table9 by scripts/make_fig_coverage.py)
# =============================================================================
import matplotlib.pyplot as plt
mets = ["f1", "mcc", "auroc"]
fig, axes = plt.subplots(len(SETTINGS), len(mets),
                         figsize=(3.1 * len(mets), 2.7 * len(SETTINGS)),
                         sharey=True, squeeze=False)
for i, (label, *_rest) in enumerate(SETTINGS):
    for j, m in enumerate(mets):
        ax = axes[i][j]
        d = cov_cplx[(cov_cplx.setting == label) & (cov_cplx.metric == m)]
        for a in ARMS + (["delong"] if m == "auroc" else []):
            dm = d[d.method == a].sort_values("n")
            if len(dm):
                ax.plot(dm.n, dm.coverage, marker="o", ms=3.5, lw=1.2, label=a)
        ax.axhline(1 - ALPHA, color="k", ls="--", lw=1)
        ax.set_xscale("log")
        if i == 0:
            ax.set_title(m)
        if i == len(SETTINGS) - 1:
            ax.set_xlabel("$n$")
        if j == 0:
            ax.set_ylabel(f"{label}\nempirical coverage", fontsize=8)
axes[0][-1].legend(fontsize=7, loc="lower right")
fig.suptitle("Coverage for metrics with no closed-form interval "
             f"(M = {ADDON['M']}, B = {ADDON['B']})", y=1.02)
fig.savefig(OUT / "figures" / "fig8_coverage_complex.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT / "figures" / "fig8_coverage_complex.pdf", bbox_inches="tight")
plt.close(fig)
print("\nsaved figures/fig8_coverage_complex.png|pdf")

# =============================================================================
# Refresh the exported artefacts so the new tables ship with the archive
# =============================================================================
RESULTS["addon_config"] = ADDON

def _tex(df, caption, label):
    return ("\\begin{table}[!t]\n\\centering\n\\caption{%s}\n\\label{%s}\n%s"
            "\\end{table}\n" % (caption, label,
                                 df.to_latex(index=False, escape=True,
                                             float_format=lambda v: "%.4f" % v)))

with open(OUT / "tables_ieee.tex", "a", encoding="utf-8") as fh:
    fh.write("\n" + _tex(
        cov_cplx.pivot_table(index=["setting", "metric", "n"], columns="method",
                             values="coverage").reset_index(),
        "Empirical coverage of nominal 95\\% intervals for metrics with no "
        "closed-form construction.", "tab:coverage_complex"))
    fh.write("\n" + _tex(
        pair_cov[["setting", "metric", "n", "true_delta", "coverage_paired",
                  "coverage_unpaired", "width_ratio", "boot_corr_AB",
                  "rejection_rate"]],
        "Coverage of the interval for a difference between two correlated models.",
        "tab:paired_coverage"))
    fh.write("\n" + _tex(
        tab11[["dataset", "A", "B", "metric", "delta", "ci", "ASL_p",
               "closed_form_alternative"]],
        "Paired differences on metrics without a closed-form interval.",
        "tab:paired_complex"))

class _NpEnc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.integer):  return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.bool_):    return bool(o)
        if isinstance(o, np.ndarray):  return o.tolist()
        return super().default(o)

(OUT / "results_all.json").write_text(json.dumps(RESULTS, indent=2, cls=_NpEnc),
                                      encoding="utf-8")
import shutil
shutil.rmtree(OUT / "medmnist", ignore_errors=True)
archive = shutil.make_archive("bootstrap_ci_results", "zip", root_dir=str(OUT))
print(f"\narchive refreshed: {archive}")
try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print(f"(download from the file browser: {e})")

print("\nSection 16 finished. New artefacts: table9_coverage_complex.csv, "
      "table10_paired_coverage.csv,\ntable11_paired_complex.csv, "
      "figures/fig8_coverage_complex.*, plus three appended LaTeX tables.")


setting: moderate (AUROC~0.85)  |  true accuracy=0.7734, f1=0.7319, mcc=0.5389, balanced_accuracy=0.7734, auroc=0.8556
  n=  50 (1000 usable cohorts, 16.7s): f1/bca=0.952  mcc/bca=0.949  auroc/bca=0.950
  n= 100 (1000 usable cohorts, 25.8s): f1/bca=0.962  mcc/bca=0.955  auroc/bca=0.959
  n= 200 (1000 usable cohorts, 49.3s): f1/bca=0.950  mcc/bca=0.949  auroc/bca=0.946
  n= 500 (1000 usable cohorts, 133.8s): f1/bca=0.948  mcc/bca=0.954  auroc/bca=0.958

setting: high (AUROC~0.97)  |  true accuracy=0.9115, f1=0.8918, mcc=0.8175, balanced_accuracy=0.9115, auroc=0.9719
  n=  50 (1000 usable cohorts, 16.5s): f1/bca=0.968  mcc/bca=0.959  auroc/bca=0.917
  n= 100 (1000 usable cohorts, 25.5s): f1/bca=0.966  mcc/bca=0.961  auroc/bca=0.947
  n= 200 (1000 usable cohorts, 45.7s): f1/bca=0.954  mcc/bca=0.948  auroc/bca=0.954
  n= 500 (1000 usable cohorts, 133.2s): f1/bca=0.943  mcc/bca=0.950  auroc/bca=0.951

Part A finished in 446s

TABLE 9 — empirical coverage of nominal 95% intervals for metric


Section 16 finished. New artefacts: table9_coverage_complex.csv, table10_paired_coverage.csv,
table11_paired_complex.csv, figures/fig8_coverage_complex.*, plus three appended LaTeX tables.
